<a href="https://colab.research.google.com/github/baker-ck/Care2ClaimZA-LLM/blob/main/Care2ClaimLLM_ZA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Care2ClaimZA-LLM**

By Mr. Clayton Baker (Lecturer: Computer Science, University of the Western Cape)

## 1. Overview

Code for data preparation, instruction tuning, training and evaluation of the full Care2ClaimZA-LLM AI.


## 2. Connect to Google Drive

Run the following cells to connect to Google Drive and set the persistent project directory.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
%cd /content/drive/My Drive/Colab Notebooks/AI-research-foundations/08

/content/drive/My Drive/Colab Notebooks/AI-research-foundations/08


## 3. Install and load packages

Run the following cell to import the required packages.


In [ ]:
%%capture
!pip install "git+https://github.com/google-deepmind/ai-foundations.git@main"
!pip install rapidfuzz
!pip install -q rouge-score

In [ ]:
%%capture
import collections
from collections import Counter, defaultdict
import glob
import io
import json
import os
import random
import re
import textwrap
from textwrap import fill
import unicodedata
from typing import Any, Dict, List, Tuple

# Google Colab Specific Utilities
try:
    from google.colab import drive
    from google.colab import userdata
except ImportError:
    drive = None
    userdata = None

# Backend Environment & Optimization Flags
os.environ["KERAS_BACKEND"] = "jax"
os.environ["XLA_PYTHON_CLIENT_MEM_FRACTION"] = "0.95"

# Fetch Kaggle API Credentials Securely from Colab Secrets
if userdata is not None:
    try:
        os.environ["KAGGLE_USERNAME"] = userdata.get("KAGGLE_USERNAME")
        os.environ["KAGGLE_KEY"] = userdata.get("KAGGLE_KEY")
    except Exception:
        pass

# Data Processing, Matching & External Datasets
import datasets
from datasets import load_dataset
import kagglehub
import numpy as np
import pandas as pd
from rapidfuzz import fuzz
from tqdm import tqdm

# Scikit-Learn Machine Learning & Evaluation Metrics
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    precision_score,
    recall_score,
)
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import train_test_split

# Multi-Turn NL Evaluation (ROUGE-L, BLEU & Semantic Similarity)
import nltk
from nltk.translate.bleu_score import SmoothingFunction, sentence_bleu
from rouge_score import rouge_scorer
from sentence_transformers import SentenceTransformer

# Download NLTK Tokenizers Quietly
nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)

# Visualisation & Plotting
import matplotlib.pyplot as plt

# Deep Learning Frameworks (JAX & KerasNLP)
import jax.numpy as jnp
import keras
import keras_nlp

import nltk
import numpy as np
import pandas as pd
from nltk.translate.bleu_score import SmoothingFunction, sentence_bleu
from rouge_score import rouge_scorer
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)

## 4.1 Dataset preparation: Task 1

Description
- Patient symptom reporting for **conversation**

Dataset source
- Kaggle

Dataset name
- healthcare-dataset-for-chatbot



In [ ]:
# ==============================================================================
# Task 1: Multi-Turn Conversation Data Preparation (3-Turn Care2ClaimLLM-ZA Schema)
# ==============================================================================

# 1. Download Healthcare Dataset from Kaggle
print("Downloading healthcare dataset from Kaggle...")
dataset_dir = kagglehub.dataset_download("rudravpatel/healthcare-dataset-for-chatbot")
json_files = [f for f in os.listdir(dataset_dir) if f.endswith(".json")]
json_path = os.path.join(dataset_dir, json_files[0])

with open(json_path, "r", encoding="utf-8") as f:
    raw_data = json.load(f)

diseases_list = raw_data.get("diseases", raw_data)
if isinstance(diseases_list, dict):
    diseases_list = list(diseases_list.values())

symptom_records = []
for item in diseases_list:
    if isinstance(item, dict):
        disease_name = item.get("name", item.get("disease", "General Condition"))
        text = None
        for key in ["symptoms", "symptom", "description", "questions", "patient", "utterances", "overview", "text"]:
            if key in item and item[key]:
                val = item[key]
                text = ", ".join(val) if isinstance(val, list) else str(val)
                break
        if text:
            symptom_records.append({"disease": disease_name, "raw_symptoms": text})

print(f"Extracted {len(symptom_records)} clinical disease records.")

# 2. System Initiation Prompt & Demographics (Ages 18-118)
SYSTEM_PROMPT = (
    "You are Care2ClaimLLM-ZA, an empathetic South African AI healthcare and coverage decision guide. "
    "Your mission is to support South Africans experiencing medical distress and health anxiety with clarity and dignity. "
    "Initiate every interaction with a warm, supportive check-in to hear how the patient is doing. "
    "Gently gather essential intake details—age, gender, medical scheme and plan option (or cash/self-pay), "
    "approximate location (suburb and province), and pain level on a 1 to 10 scale—"
    "to provide tailored primary care triage and scheme coverage guidance without making unauthorized clinical diagnoses."
)

PATIENT_DEMOGRAPHICS = [
    (f"{age}-year-old {gender}", gender)
    for age in range(18, 90)
    for gender in ["female", "male"]
]

# Re-use Dual-Plan Scheme Registry from Task 2
SA_SCHEMES_AND_PLANS = {
    "DiscoveryHMS": {
        "KeyCare_Access": "Discovery KeyCare Access (Entry Network Option)",
        "Executive_Comprehensive": "Discovery Executive Comprehensive (Top-Tier Plan)"
    },
    "BonitasMF": {
        "BonStart_Essential": "Bonitas BonStart (Basic Entry Network Option)",
        "BonComprehensive": "Bonitas BonComprehensive (Full Comprehensive Cover)"
    },
    "MomentumMS": {
        "Ingwe_Network": "Momentum Ingwe Option (Entry Level Network)",
        "Summit_Extensive": "Momentum Summit Option (Top-Tier Extensive Cover)"
    },
    "BestMedMS": {
        "Beat_Network": "Bestmed Beat Series (Basic Network Option)",
        "Pace_Comprehensive": "Bestmed Pace Series (Comprehensive Plan)"
    },
    "FedHealthMS": {
        "flexiFED_Essential": "Fedhealth flexiFED (Entry Network Option)",
        "maxiFED_Comprehensive": "Fedhealth maxiFED (Top-Tier Comprehensive Plan)"
    },
    "Cash_Only_SelfPay": {
        "SelfPay_Direct": "Cash Only / Direct Self-Pay (Uninsured / Non-Scheme)"
    }
}

SCHEME_PLAN_REGISTRY = []
for s_key, plans in SA_SCHEMES_AND_PLANS.items():
    for p_key, p_name in plans.items():
        SCHEME_PLAN_REGISTRY.append((s_key, p_key, p_name))


# 3. Multilingual Dictionaries & Translation Engine
SA_GREETINGS = {
    "isiZulu": {"singular": ["Sawubona", "Yebo sawubona", "Sanibonani"], "plural": ["Sanibona", "Sanibonani ma-Afrika", "Ngiya nibulisa"]},
    "isiXhosa": {"singular": ["Molo", "Molo dokotela", "Molo sisi"], "plural": ["Molweni", "Molweni ma-Kwayi", "Molweni miphakathi"]},
    "Afrikaans": {"singular": ["Goeie môre", "Goeiedag", "Hallo", "Goeienaand"], "plural": ["Goeiedag almal", "Hallo julle", "Goeie môre julle"]},
    "English_SA": {"singular": ["Hello doctor", "Hi sister", "Good morning doc", "Howzit"], "plural": ["Hello everyone", "Hi team", "Good day doctors"]},
    "Sepedi": {"singular": ["Thobela", "Agee", "Dumela"], "plural": ["Thobelang", "Ageeng", "Dumelang"]},
    "Sesotho": {"singular": ["Dumela", "Lumela", "Khotso"], "plural": ["Dumelang", "Lumelang", "Khotsoaneng"]},
    "Setswana": {"singular": ["Dumela", "Dumela rre", "Dumela mme"], "plural": ["Dumelang", "Dumelang bagaetsho"]},
    "siSwati": {"singular": ["Sawubona", "Yebo sawubona"], "plural": ["Sanibonani", "Sanibona bekunene"]},
    "Tshivenda": {"singular": ["Ndaa", "Aa", "Ndaa dokotela"], "plural": ["Ndaani", "Aani", "Vho-Ndaa"]},
    "isiNdebele": {"singular": ["Lotjhani", "Lotjha"], "plural": ["Lotjhanini", "Lotjhanini nina boke"]},
    "Xitsonga": {"singular": ["Avuxeni", "Minjhani"], "plural": ["Avuxeni hinkwenu", "Minjhanini"]},
    "Urban_Slang": ["Howzit", "Aweh", "Aweh doc", "Heita", "Heita da", "Sho", "Sho magents", "Ola", "Ola doc", "Yes", "Hey", "Hiya"]
}

SA_CHECK_INS = [
    "How are you?", "How's it going doc?", "What's up?", "Kanjani?",
    "O kae?", "Hoe gaan dit?", "Unjani?", "How's the body?",
    "Unjani doctor?", "Hoe lijkit?", "Ndi khou vhisa, unjani?"
]

SA_EXCLAMATIONS = [
    "Sjoe!", "Joh!", "Sho!", "Eish!", "Jinne!", "Yoh!",
    "Lekker", "Lekker man", "Aweh", "Grootman", "Tyhini!",
    "Batho ba Modimo!", "Mamma mia!", "Hayibo!"
]

SA_CLOSINGS = [
    "Thanks doc, sharp! Must I go to the local clinic or get a pre-auth from my medical scheme?",
    "Thank you sister. Should I visit the GP today or just get something at the chemist?",
    "Siyabonga doctor, I will head to the clinic right away.",
    "Baie dankie doc! Will my medical scheme cover this GP visit under Prescribed Minimum Benefits (PMB)?",
    "Enkosi kakhulu sister! Do I need a referral from the clinic before going to casualty?"
]

SA_PAIN_DESCRIPTIONS = [
    ("No pain at all doc, 0/10, just feeling generally unwell.", 0), ("0 out of 10 pain hey, no sore places, just very tired and feverish.", 0),
    ("Ja doc, zero pain, but I noticed a strange rash on my arm.", 0), ("Pain is 0/10, I'm just calling to check my scheme benefits.", 0),
    ("Barely notice it, like a 1/10, just a slight flutter in my stomach.", 1), ("1 out of 10, very mild discomfort, hardly bothering me.", 1),
    ("Eish, 2/10, just a small discomfort but wanted to be safe.", 2), ("Maybe a 2 out of 10, minor ache now and then when I walk.", 2),
    ("Eish it's like a 3/10, just nagging me in the background since morning.", 3), ("3 out of 10 doc, noticeable but tolerable without pills.", 3),
    ("Ja sister, it's mild like 4/10, but my body feels so heavy.", 4), ("4 out of 10 pain, annoying and making it hard to concentrate.", 4),
    ("It's a dull ache, maybe a 5/10, but it just won't go away.", 5), ("5 out of 10, halfway up the scale, definitely sore today.", 5),
    ("Yo doc, it's burning me inside, maybe a 6/10 when I breathe in.", 6), ("6 out of 10, really uncomfortable now, can't ignore it at all.", 6),
    ("Ja doc, it's about a 7/10. Very sore when I try to move around.", 7), ("Sjoe doc, 7/10 stabbing pain every time I try to cough.", 7),
    ("Eish, the pain is like an 8 out of 10, it's really thumping me.", 8), ("Sho, 8/10! The pain is biting real hard, eish I can't take it.", 8),
    ("Sho, it's sharp hey! Maybe 9 out of 10, couldn't sleep last night.", 9), ("Haibo, 9/10! It feels like a hammer throbbing in my head!", 9),
    ("Jinne doc, 10 out of 10! I can't even stand up, it's excruciating!", 10), ("10/10 worst pain of my life! Need emergency casualty right now!", 10)
]

SA_PROVINCIAL_SUBURBS = {
    "Gauteng": ["Soweto", "Sandton", "Alexandra", "Tembisa", "Centurion", "Mamelodi", "Diepsloot", "Orange Farm", "Katlehong", "Vosloorus"],
    "Western_Cape": ["Khayelitsha", "Mitchells Plain", "Du Noon", "Bellville", "Gugulethu", "Langa", "Sea Point", "Delft", "Nyanga", "Kraaifontein"],
    "KwaZulu_Natal": ["Umlazi", "Phoenix", "Chatsworth", "KwaMashu", "Ballito", "Pietermaritzburg", "Madadeni", "Inanda", "Empangeni"],
    "Eastern_Cape": ["Mdantsane", "Motherwell", "Ibhayi", "Zwide", "Mthatha", "Gcuwa (Butterworth)", "Makhanda", "Gqeberha"],
    "Limpopo": ["Seshego", "Turfloop (Mankweng)", "Thohoyandou", "Tzaneen", "Lebowakgomo", "Giyani", "Polokwane", "Musina"],
    "Mpumalanga": ["Emalahleni (Witbank)", "Mbombela (Nelspruit)", "Kanyamazane", "Secunda", "Middelburg", "Bushbuckridge"],
    "North_West": ["Tlhabane (Rustenburg)", "Mahikeng", "Potchefstroom", "Klerksdorp", "Brits", "Jouberton"],
    "Free_State": ["Mangaung (Bloemfontein)", "Botshabelo", "Thaba 'Nchu", "Thabong (Welkom)", "Harrismith", "Sasolburg"],
    "Northern_Cape": ["Galeshewe (Kimberley)", "Paballelo (Upington)", "Kuruman", "Springbok", "De Aar", "Kathu"]
}

SA_TRANSLATION_MAP = {
    "sawubona": "hello", "sanibonani": "hello", "molo": "hello", "molweni": "hello",
    "dumela": "hello", "dumelang": "hello", "thobela": "hello", "avuxeni": "hello",
    "lotjhani": "hello", "ndaa": "hello", "goeie môre": "good morning", "goeiedag": "good day",
    "howzit": "hello", "heita": "hello", "aweh": "hello", "yebo": "yes", "ja": "yes",
    "ee": "yes", "ndza": "yes", "sharp": "okay fine", "kwaai": "good", "hayi": "no",
    "nee": "no", "awa": "no", "che": "no", "haibo": "no way", "aikona": "no way",
    "eish": "", "sjoe": "", "joh": "", "jinne": "", "yoh": "", "tyhini": "",
    "udokotela": "doctor", "ugqirha": "doctor", "ngaka": "doctor", "dokodela": "doctor",
    "unesi": "nurse", "mooki": "nurse", "verpleegster": "nurse", "suster": "nurse",
    "isibhedlela": "hospital", "sepetlele": "hospital", "hospitaal": "hospital",
    "isangoma": "traditional healer", "inyanga": "traditional healer", "chemist": "pharmacy",
    "biting": "cramping ache", "thumping": "throbbing pain", "gogo": "elderly patient"
}

def translate_and_normalize_user_turn(text: str) -> str:
    text_clean = text.lower()
    for slang, standard in SA_TRANSLATION_MAP.items():
        if slang in text_clean:
            text_clean = re.sub(r'\b' + re.escape(slang) + r'\b', standard, text_clean)
    return re.sub(r'\s+', ' ', text_clean).strip()

# 4. Emergency engine

NATIONAL_EMERGENCY_NUMBERS = {
    "Mobile_Universal": {"number": "112", "label": "National Mobile Emergency Hotline (Toll-Free)", "link": "tel:112"},
    "SAPS_Police": {"number": "10111", "label": "SAPS Crime & Theft Emergency", "link": "tel:10111"},
    "National_Ambulance": {"number": "10177", "label": "National Public Ambulance & Fire", "link": "tel:10177"},
    "Netcare_911": {"number": "082911", "label": "Netcare 911 Private Ambulance", "link": "tel:082911"},
    "ER24": {"number": "084124", "label": "ER24 Private Emergency Services", "link": "tel:084124"}
}

SA_PROVINCIAL_EMERGENCY_CONTACTS = {
    "Gauteng": {
        "Metro_Emergency": {"number": "0113755911", "label": "Johannesburg Metro Emergency Services", "link": "tel:0113755911"},
        "Tshwane_Fire_Ambulance": {"number": "0123106200", "label": "Pretoria / Tshwane Emergency Services", "link": "tel:0123106200"},
        "Ekurhuleni_Control": {"number": "0114580911", "label": "Ekurhuleni East Rand Emergency Centre", "link": "tel:0114580911"},
        "Provincial_EMS": {"number": "10177", "label": "Gauteng Health Provincial Ambulance", "link": "tel:10177"}
    },
    "Western_Cape": {
        "City_Metro_Police": {"number": "0214807700", "label": "Cape Town Metro Emergency & Crime Line", "link": "tel:0214807700"},
        "WC_Provincial_EMS": {"number": "0219370500", "label": "Western Cape Provincial Ambulance Dispatch", "link": "tel:0219370500"},
        "Disaster_Management": {"number": "0800225510", "label": "Western Cape Disaster Control", "link": "tel:0800225510"}
    },
    "KwaZulu_Natal": {
        "Durban_Metro_Police": {"number": "0313610000", "label": "eThekwini / Durban Metro Emergency", "link": "tel:0313610000"},
        "KZN_Provincial_EMS": {"number": "0338467200", "label": "KZN Health Emergency Operations Centre", "link": "tel:0338467200"},
        "Pietermaritzburg_Fire": {"number": "0338455911", "label": "Msunduzi Fire & Ambulance", "link": "tel:0338455911"}
    },
    "Eastern_Cape": {
        "Nelson_Mandela_Bay": {"number": "0415851555", "label": "Gqeberha / NMB Metro Disaster Centre", "link": "tel:0415851555"},
        "EC_Provincial_EMS": {"number": "0437031500", "label": "Eastern Cape Provincial Ambulance Control", "link": "tel:0437031500"},
        "Buffalo_City_Fire": {"number": "0437059000", "label": "East London / Buffalo City Emergency", "link": "tel:0437059000"}
    },
    "Limpopo": {
        "Polokwane_Disaster": {"number": "0152902000", "label": "Polokwane Emergency Control Room", "link": "tel:0152902000"},
        "Limpopo_EMS": {"number": "0152936000", "label": "Limpopo Provincial Ambulance Services", "link": "tel:0152936000"}
    },
    "Mpumalanga": {
        "Mbombela_Emergency": {"number": "0137592129", "label": "Nelspruit / Mbombela Disaster Management", "link": "tel:0137592129"},
        "Mpumalanga_EMS": {"number": "0137413216", "label": "Mpumalanga Provincial Ambulance Dispatch", "link": "tel:0137413216"}
    },
    "North_West": {
        "Rustenburg_Fire": {"number": "0145903232", "label": "Rustenburg Emergency & Fire Services", "link": "tel:0145903232"},
        "NW_Provincial_EMS": {"number": "0183840240", "label": "North West Provincial Ambulance Hotline", "link": "tel:0183840240"}
    },
    "Free_State": {
        "Mangaung_Fire": {"number": "0514066666", "label": "Bloemfontein / Mangaung Emergency Centre", "link": "tel:0514066666"},
        "FS_Provincial_EMS": {"number": "0514076000", "label": "Free State Health Ambulance Dispatch", "link": "tel:0514076000"}
    },
    "Northern_Cape": {
        "Sol_Plaatje_Emergency": {"number": "0538385100", "label": "Kimberley / Sol Plaatje Emergency Services", "link": "tel:0538385100"},
        "NC_Provincial_EMS": {"number": "0538312812", "label": "Northern Cape Provincial Ambulance Hotline", "link": "tel:0538312812"}
    }
}

def generate_emergency_call_block(province: str = "Gauteng") -> str:
    """
    Generates a Markdown block with active clickable call links tailored
    to the patient's specific province and national emergency numbers.
    """
    prov_key = province.replace(" ", "_")
    prov_contacts = SA_PROVINCIAL_EMERGENCY_CONTACTS.get(prov_key, SA_PROVINCIAL_EMERGENCY_CONTACTS["Gauteng"])

    call_block = (
        "🚨 **CRITICAL MEDICAL EMERGENCY DETECTED** 🚨\n"
        "If you or the patient are experiencing severe distress, please place an immediate call:\n\n"
        "📞 **Immediate Call Links (Click to Dial):**\n"
        "• 🚑 **Mobile Universal Emergency**: [Call 112 (Free)](tel:112)\n"
        "• 🚔 **SAPS Police & Crime**: [Call 10111](tel:10111)\n"
        "• 🚑 **National Public Ambulance**: [Call 10177](tel:10177)\n"
        "• 🩺 **Netcare 911 (Private)**: [Call 082 911](tel:082911)\n"
        "• 🩺 **ER24 (Private)**: [Call 084 124](tel:084124)\n\n"
        f"🏛️ **{province.replace('_', ' ')} Regional Emergency Lines:**\n"
    )

    for key, info in prov_contacts.items():
        call_block += f"• {info['label']}: [{info['number']}]({info['link']})\n"

    return call_block

# 5. Clinical Matcher Engine
CANONICAL_BODY_REGIONS = {
    "HEAD_CRANIAL": "headache migraine throbbing head facial pressure sinus eye pain cranial vision blurred",
    "CHEST_ANTERIOR": "chest pain shortness of breath wheezing cough thoracic tightness heart palpitation difficulty breathing",
    "ABDOMEN_RLQ": "lower right side abdomen right iliac fossa appendix pain appendicitis right groin",
    "ABDOMEN_RUQ": "upper right quadrant abdomen gallbladder liver right shoulder blade pain hypochondrium",
    "ABDOMEN_CENTRAL": "stomach cramping stomachache belly pain bloating periumbilical acid reflux heartburn nausea",
    "BACK_LUMBAR": "lower back pain lumbar spine flank kidney stone backache stooped posture back stiffness",
    "JOINTS_PERIPHERAL": "joint pain stiffness swelling knee elbow ankle wrist arthritis crunching limb weakness",
    "PELVIS_GENITAL": "pelvic pain delivery pregnancy painful urination blood in urine prostate vaginal discharge",
    "GENERAL": "fatigue fever night sweats weight loss malaise body ache weakness rash skin pimples",
}

CANONICAL_TRIAGE_CATEGORIES = {
    "Requires Pre-Auth / Emergency Triage": "chest pain shortness of breath seizure stroke sudden severe bleeding fracture emergency MRI CT scan surgery operation hospital admission appendicitis",
    "Out-of-Cover (Scheme Exclusion)": "cosmetic surgery liposuction lipectomy wrinkle botox non-formulary weight loss supplement diet diary autopsy forensic chiropractic",
    "In-Cover (Primary Care / GP Consultation)": "viral upper respiratory cold flu sore throat routine physical exam well child GP consultation blood pressure check hypertension prescription refill",
}

class LightweightClinicalMatcher:
    def __init__(self, region_dict, triage_dict):
        self.region_keys = list(region_dict.keys())
        self.region_texts = [region_dict[k] for k in self.region_keys]
        self.triage_keys = list(triage_dict.keys())
        self.triage_texts = [triage_dict[k] for k in self.triage_keys]

        self.region_vec = TfidfVectorizer(ngram_range=(1, 3), analyzer="word").fit(self.region_texts)
        self.region_tfidf = self.region_vec.transform(self.region_texts)

        self.triage_vec = TfidfVectorizer(ngram_range=(1, 3), analyzer="word").fit(self.triage_texts)
        self.triage_tfidf = self.triage_vec.transform(self.triage_texts)

    def predict_region(self, patient_text: str):
        normalized = translate_and_normalize_user_turn(patient_text)
        user_vec = self.region_vec.transform([normalized])
        sims = cosine_similarity(user_vec, self.region_tfidf)[0]

        best_tfidf_idx = np.argmax(sims)
        tfidf_score = float(sims[best_tfidf_idx])
        tfidf_region = self.region_keys[best_tfidf_idx]

        best_fuzzy_score, fuzzy_region = 0, "GENERAL"
        for region, kw_text in zip(self.region_keys, self.region_texts):
            score = fuzz.token_set_ratio(normalized, kw_text)
            if score > best_fuzzy_score:
                best_fuzzy_score, fuzzy_region = score, region

        if tfidf_region == fuzzy_region:
            return tfidf_region, (tfidf_score * 0.6 + (best_fuzzy_score / 100.0) * 0.4)
        return (tfidf_region, tfidf_score) if tfidf_score >= (best_fuzzy_score / 100.0) else (fuzzy_region, best_fuzzy_score / 100.0)

    def predict_triage(self, patient_text: str):
        normalized = translate_and_normalize_user_turn(patient_text)
        user_vec = self.triage_vec.transform([normalized])
        sims = cosine_similarity(user_vec, self.triage_tfidf)[0]
        best_idx = np.argmax(sims)
        return self.triage_keys[best_idx], float(sims[best_idx])

matcher = LightweightClinicalMatcher(CANONICAL_BODY_REGIONS, CANONICAL_TRIAGE_CATEGORIES)

# 6. Multi-Turn Dialogue Synthesis

raw_chat_data = []

for idx, record in enumerate(symptom_records, start=1):
    disease = record["disease"]
    symptoms_raw = record["raw_symptoms"]
    symptoms_list = [s.strip() for s in symptoms_raw.split(",")]
    primary_symptom = symptoms_list[0] if symptoms_list else symptoms_raw
    secondary_symptom = symptoms_list[1] if len(symptoms_list) > 1 else "general malaise"

    # Normalize symptoms for TF-IDF / Fuzzy Matcher
    normalized_symptoms = translate_and_normalize_user_turn(symptoms_raw)
    body_region, _ = matcher.predict_region(normalized_symptoms)
    triage_cat, _ = matcher.predict_triage(normalized_symptoms)

    # Sample SA Demographics, Location & Scheme Plan
    province = random.choice(list(SA_PROVINCIAL_SUBURBS.keys()))
    suburb = random.choice(SA_PROVINCIAL_SUBURBS[province])
    demo_str, gender = random.choice(PATIENT_DEMOGRAPHICS)
    scheme_key, plan_key, plan_name = random.choice(SCHEME_PLAN_REGISTRY)

    lang_key = random.choice(list(SA_GREETINGS.keys()))
    greeting = random.choice(
        SA_GREETINGS[lang_key] if lang_key == "Urban_Slang"
        else SA_GREETINGS[lang_key]["singular"] + SA_GREETINGS[lang_key]["plural"]
    )
    exclamation = random.choice(SA_EXCLAMATIONS)
    check_in = random.choice(SA_CHECK_INS)
    pain_str, pain_score = random.choice(SA_PAIN_DESCRIPTIONS)
    closing = random.choice(SA_CLOSINGS)

    # Determine Emergency Status & Scheme Advice
    if "Pre-Auth" in triage_cat or pain_score >= 8:
        advice = f"This requires immediate evaluation under PMB emergency rules at an Urgent Care or Casualty unit ({plan_name})."
        facility = "the nearest Tertiary Hospital Emergency Casualty"
        emergency_links = generate_emergency_call_block(province)
    elif scheme_key == "Cash_Only_SelfPay" or "Out-of-Cover" in triage_cat:
        advice = f"Note: This service is classified as elective/lifestyle or self-pay under {plan_name} rules."
        facility = "your local retail pharmacy clinic or private GP (Self-Pay)"
        emergency_links = ""
    else:
        advice = f"You should book a consultation with your network GP or clinic sister under {plan_name} primary benefits."
        facility = "your designated network day clinic or GP"
        emergency_links = ""

    # Turn 1: User initiation & Assistant Intake Request
    t1_user_raw = f"{greeting}! {exclamation} {check_in} I'm writing from {suburb} in {province.replace('_', ' ')}. I'm experiencing {primary_symptom.lower()}."
    t1_user_translated = translate_and_normalize_user_turn(t1_user_raw)

    t1_assistant = (
        f"Dumela / Hello! I am Care2ClaimLLM-ZA, your healthcare and coverage decision guide. "
        f"I hear that you are not feeling well in {suburb}, and I am here to help you navigate this with care and dignity.\n\n"
        f"To give you the best guidance, could you please share a few quick details?\n"
        f"1. Your age and gender\n"
        f"2. Your medical scheme and plan option (or if you are self-paying/cash)\n"
        f"3. How long you've felt this {primary_symptom.lower()} and if you have other symptoms like {secondary_symptom.lower()}\n"
        f"4. Your current pain level on a scale from 1 to 10"
    )

    # Turn 2: User details (including Scheme) & Assistant Scheme-Aware Triage
    t2_user_raw = (
        f"I am a {demo_str}. My medical cover is {plan_name}. "
        f"Eish, it started about 3 days ago, and ja, I also have {secondary_symptom.lower()}. {pain_str}"
    )
    t2_user_translated = translate_and_normalize_user_turn(t2_user_raw)

    t2_assistant_parts = []
    if emergency_links:
        t2_assistant_parts.append(emergency_links)

    t2_assistant_parts.append(
        f"Thank you for sharing that. Based on your symptoms of {primary_symptom.lower()} and {secondary_symptom.lower()} "
        f"(Severity: {pain_score}/10) for a {demo_str} in {suburb} on {plan_name}:\n\n"
        f"• **Clinical Assessment Summary**: Suspected {disease} ({body_region}).\n"
        f"• **Medical Scheme / Triage Category**: {triage_cat}.\n"
        f"• **Scheme Plan Option**: {plan_name}.\n"
        f"• **Recommended Action**: {advice}\n"
        f"• **Suggested Facility Type**: {facility}."
    )
    t2_assistant = "\n\n".join(t2_assistant_parts)

    # Turn 3: Wrap-up & Scheme Advice
    t3_user_raw = closing
    t3_user_translated = translate_and_normalize_user_turn(t3_user_raw)

    t3_assistant = (
        f"Kua right / Keep safe! If you decide to visit {facility}, remember to take your ID and {scheme_key} card if applicable. "
        f"Please reach out immediately or call emergency services if your symptoms worsen!"
    )

    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": t1_user_raw},
        {"role": "assistant", "content": t1_assistant},
        {"role": "user", "content": t2_user_raw},
        {"role": "assistant", "content": t2_assistant},
        {"role": "user", "content": t3_user_raw},
        {"role": "assistant", "content": t3_assistant},
    ]

    record_entry = {
        "case_id": f"SA-CASE-{idx:04d}",
        "messages": messages,
        "translated_user_turns": [t1_user_translated, t2_user_translated, t3_user_translated],
        "meta": {
            "topic": disease,
            "region": body_region,
            "triage_category": triage_cat,
            "scheme_key": scheme_key,
            "plan_key": plan_key,
            "suburb": suburb,
            "province": province,
            "pain_score": pain_score,
            "patient_gender": gender,
        },
    }
    raw_chat_data.append(record_entry)



# 7. Deduplication & Equalised Class Balancing
seen = {}
for d in raw_chat_data:
    user_turns = [m["content"] for m in d["messages"] if m["role"] == "user"]
    key = "|".join(user_turns)
    seen[key] = d

chat_data = list(seen.values())

triage_groups = defaultdict(list)
for record in chat_data:
    cat = record["meta"]["triage_category"]
    triage_groups[cat].append(record)

target_per_gender = 40  # 40 Female + 40 Male = 80 per triage category (240 total)
balanced_chat_data = []

random.seed(42)  # Fixed seed for perfect reproducibility

for triage_cat, records in triage_groups.items():
    female_records = [r for r in records if r["meta"]["patient_gender"] == "female"]
    male_records = [r for r in records if r["meta"]["patient_gender"] == "male"]

    sampled_female = random.sample(female_records, k=min(target_per_gender, len(female_records)))
    sampled_male = random.sample(male_records, k=min(target_per_gender, len(male_records)))

    balanced_chat_data.extend(sampled_female + sampled_male)

random.shuffle(balanced_chat_data)

# 8. Save to JSONL & Final Distribution Report
output_jsonl = "task1_conversation_data_balanced.jsonl"
with open(output_jsonl, "w", encoding="utf-8") as f:
    for entry in balanced_chat_data:
        f.write(json.dumps(entry, ensure_ascii=False) + "\n")

triage_counts = Counter(d["meta"]["triage_category"] for d in balanced_chat_data)
region_counts = Counter(d["meta"]["region"] for d in balanced_chat_data)
province_counts = Counter(d["meta"]["province"] for d in balanced_chat_data)
pain_counts = Counter(d["meta"]["pain_score"] for d in balanced_chat_data)
gender_counts = Counter(d["meta"]["patient_gender"] for d in balanced_chat_data)

print(f"\n{'=' * 60}")
print("  TASK 1: BALANCED MULTI-TURN DATASET GENERATION REPORT")
print(f"{'=' * 60}")
print(f"Total Exported Multi-Turn Conversations: {len(balanced_chat_data)}")
print(f"Saved File: {output_jsonl}\n")

print("--- 1. Scheme Triage Category Breakdown ---")
for triage, count in sorted(triage_counts.items()):
    pct = (count / len(balanced_chat_data)) * 100
    print(f"  • {triage:45s}: {count:3d} ({pct:5.1f}%)")

print("\n--- 2. Anatomical Body Region Breakdown ---")
for region, count in sorted(region_counts.items()):
    pct = (count / len(balanced_chat_data)) * 100
    print(f"  • {region:25s}: {count:3d} ({pct:5.1f}%)")

print("\n--- 3. Provincial Geographic Breakdown ---")
for province, count in sorted(province_counts.items()):
    pct = (count / len(balanced_chat_data)) * 100
    print(f"  • {province:20s}: {count:3d} ({pct:5.1f}%)")

print("\n--- 4. Pain Scale Breakdown (0-10) ---")
for score in range(11):
    count = pain_counts.get(score, 0)
    pct = (count / len(balanced_chat_data)) * 100
    print(f"  • Pain Rating {score:2d}/10: {count:3d} ({pct:5.1f}%)")

print("\n--- 5. Gender Demographic Split ---")
for gender, count in sorted(gender_counts.items()):
    pct = (count / len(balanced_chat_data)) * 100
    print(f"  • {gender.title():10s}: {count:3d} ({pct:5.1f}%)")
print(f"{'=' * 60}\n")

Extracted 275 clinical disease records.

  TASK 1: BALANCED MULTI-TURN DATASET GENERATION REPORT
Total Exported Multi-Turn Conversations: 140
Saved File: task1_conversation_data_balanced.jsonl

--- 1. Scheme Triage Category Breakdown ---
  • In-Cover (Primary Care / GP Consultation)    :  22 ( 15.7%)
  • Out-of-Cover (Scheme Exclusion)              :  38 ( 27.1%)
  • Requires Pre-Auth / Emergency Triage         :  80 ( 57.1%)

--- 2. Anatomical Body Region Breakdown ---
  • ABDOMEN_CENTRAL          :   1 (  0.7%)
  • ABDOMEN_RLQ              :  17 ( 12.1%)
  • ABDOMEN_RUQ              :   1 (  0.7%)
  • BACK_LUMBAR              :   5 (  3.6%)
  • CHEST_ANTERIOR           :  37 ( 26.4%)
  • GENERAL                  :  43 ( 30.7%)
  • HEAD_CRANIAL             :  13 (  9.3%)
  • JOINTS_PERIPHERAL        :  18 ( 12.9%)
  • PELVIS_GENITAL           :   5 (  3.6%)

--- 3. Provincial Geographic Breakdown ---
  • Eastern_Cape        :  17 ( 12.1%)
  • Free_State          :  20 ( 14.3%)
  • Gau

## 4.2 Dataset preparation: Task 2
Description
- patient claims & consultation text for coverage **classification**

Dataset source
- HuggingFace

Dataset name
- medical_mtsamples



In [ ]:
# ==============================================================================
# Task 2: Classification Data Preparation (MTSamples -> 3-Class Policy Mapping)
# ==============================================================================

# ==============================================================================
# STEP 1: DOWNLOAD DATASET FROM HUGGING FACE
# ==============================================================================
print("Downloading MTSamples dataset from Hugging Face...")
mt_data = load_dataset("NickyNicky/medical_mtsamples", split="train")
df_raw = pd.DataFrame(mt_data)[["medical_specialty", "description"]].dropna()
df_raw["medical_specialty"] = df_raw["medical_specialty"].str.strip()
df_raw["description"] = df_raw["description"].str.strip()

# ==============================================================================
# STEP 2: DEFINE SA MEDICAL SCHEMES & DUAL-PLAN DICTIONARIES
# ==============================================================================
SA_SCHEMES_AND_PLANS = {
    "DiscoveryHMS": {
        "KeyCare_Access": "Discovery KeyCare Access (Entry Network Option)",
        "Executive_Comprehensive": "Discovery Executive Comprehensive (Top-Tier Plan)"
    },
    "BonitasMF": {
        "BonStart_Essential": "Bonitas BonStart (Basic Entry Network Option)",
        "BonComprehensive": "Bonitas BonComprehensive (Full Comprehensive Cover)"
    },
    "MomentumMS": {
        "Ingwe_Network": "Momentum Ingwe Option (Entry Level Network)",
        "Summit_Extensive": "Momentum Summit Option (Top-Tier Extensive Cover)"
    },
    "BestMedMS": {
        "Beat_Network": "Bestmed Beat Series (Basic Network Option)",
        "Pace_Comprehensive": "Bestmed Pace Series (Comprehensive Plan)"
    },
    "FedHealthMS": {
        "flexiFED_Essential": "Fedhealth flexiFED (Entry Network Option)",
        "maxiFED_Comprehensive": "Fedhealth maxiFED (Top-Tier Comprehensive Plan)"
    },
    "Cash_Only_SelfPay": {
        "SelfPay_Direct": "Cash Only / Direct Self-Pay (Uninsured / Non-Scheme)"
    }
}

# Flatten list of (scheme_key, plan_key, plan_name) for uniform sampling
SCHEME_PLAN_REGISTRY = []
for s_key, plans in SA_SCHEMES_AND_PLANS.items():
    for p_key, p_name in plans.items():
        SCHEME_PLAN_REGISTRY.append((s_key, p_key, p_name))

# Contextual SA Dictionaries (Reused from Task 1)
SA_GREETINGS = {
    "isiZulu": {"singular": ["Sawubona", "Yebo sawubona", "Sanibonani"], "plural": ["Sanibona", "Sanibonani ma-Afrika", "Ngiya nibulisa"]},
    "isiXhosa": {"singular": ["Molo", "Molo dokotela", "Molo sisi"], "plural": ["Molweni", "Molweni ma-Kwayi", "Molweni miphakathi"]},
    "Afrikaans": {"singular": ["Goeie môre", "Goeiedag", "Hallo", "Goeienaand"], "plural": ["Goeiedag almal", "Hallo julle", "Goeie môre julle"]},
    "English_SA": {"singular": ["Hello doctor", "Hi sister", "Good morning doc", "Howzit"], "plural": ["Hello everyone", "Hi team", "Good day doctors"]},
    "Sepedi": {"singular": ["Thobela", "Agee", "Dumela"], "plural": ["Thobelang", "Ageeng", "Dumelang"]},
    "Sesotho": {"singular": ["Dumela", "Lumela", "Khotso"], "plural": ["Dumelang", "Lumelang", "Khotsoaneng"]},
    "Setswana": {"singular": ["Dumela", "Dumela rre", "Dumela mme"], "plural": ["Dumelang", "Dumelang bagaetsho"]},
    "siSwati": {"singular": ["Sawubona", "Yebo sawubona"], "plural": ["Sanibonani", "Sanibona bekunene"]},
    "Tshivenda": {"singular": ["Ndaa", "Aa", "Ndaa dokotela"], "plural": ["Ndaani", "Aani", "Vho-Ndaa"]},
    "isiNdebele": {"singular": ["Lotjhani", "Lotjha"], "plural": ["Lotjhanini", "Lotjhanini nina boke"]},
    "Xitsonga": {"singular": ["Avuxeni", "Minjhani"], "plural": ["Avuxeni hinkwenu", "Minjhanini"]},
    "Urban_Slang": ["Howzit", "Aweh", "Aweh doc", "Heita", "Heita da", "Sho", "Sho magents", "Ola", "Ola doc", "Yes", "Hey", "Hiya"]
}

SA_EXCLAMATIONS = [
    "Sjoe!", "Joh!", "Sho!", "Eish!", "Jinne!", "Yoh!",
    "Lekker", "Lekker man", "Aweh", "Grootman", "Tyhini!",
    "Batho ba Modimo!", "Mamma mia!", "Hayibo!"
]

SA_PAIN_DESCRIPTIONS = [
    ("No pain at all doc, 0/10, just feeling generally unwell.", 0), ("0 out of 10 pain hey, no sore places, just very tired and feverish.", 0),
    ("Ja doc, zero pain, but I noticed a strange rash on my arm.", 0), ("Pain is 0/10, I'm just calling to check my scheme benefits.", 0),
    ("Barely notice it, like a 1/10, just a slight flutter in my stomach.", 1), ("1 out of 10, very mild discomfort, hardly bothering me.", 1),
    ("Eish, 2/10, just a small discomfort but wanted to be safe.", 2), ("Maybe a 2 out of 10, minor ache now and then when I walk.", 2),
    ("Eish it's like a 3/10, just nagging me in the background since morning.", 3), ("3 out of 10 doc, noticeable but tolerable without pills.", 3),
    ("Ja sister, it's mild like 4/10, but my body feels so heavy.", 4), ("4 out of 10 pain, annoying and making it hard to concentrate.", 4),
    ("It's a dull ache, maybe a 5/10, but it just won't go away.", 5), ("5 out of 10, halfway up the scale, definitely sore today.", 5),
    ("Yo doc, it's burning me inside, maybe a 6/10 when I breathe in.", 6), ("6 out of 10, really uncomfortable now, can't ignore it at all.", 6),
    ("Ja doc, it's about a 7/10. Very sore when I try to move around.", 7), ("Sjoe doc, 7/10 stabbing pain every time I try to cough.", 7),
    ("Eish, the pain is like an 8 out of 10, it's really thumping me.", 8), ("Sho, 8/10! The pain is biting real hard, eish I can't take it.", 8),
    ("Sho, it's sharp hey! Maybe 9 out of 10, couldn't sleep last night.", 9), ("Haibo, 9/10! It feels like a hammer throbbing in my head!", 9),
    ("Jinne doc, 10 out of 10! I can't even stand up, it's excruciating!", 10), ("10/10 worst pain of my life! Need emergency casualty right now!", 10)
]

SA_PROVINCIAL_SUBURBS = {
    "Gauteng": ["Soweto", "Sandton", "Alexandra", "Tembisa", "Centurion", "Mamelodi", "Diepsloot", "Orange Farm", "Katlehong", "Vosloorus"],
    "Western_Cape": ["Khayelitsha", "Mitchells Plain", "Du Noon", "Bellville", "Gugulethu", "Langa", "Sea Point", "Delft", "Nyanga", "Kraaifontein"],
    "KwaZulu_Natal": ["Umlazi", "Phoenix", "Chatsworth", "KwaMashu", "Ballito", "Pietermaritzburg", "Madadeni", "Inanda", "Empangeni"],
    "Eastern_Cape": ["Mdantsane", "Motherwell", "Ibhayi", "Zwide", "Mthatha", "Gcuwa (Butterworth)", "Makhanda", "Gqeberha"],
    "Limpopo": ["Seshego", "Turfloop (Mankweng)", "Thohoyandou", "Tzaneen", "Lebowakgomo", "Giyani", "Polokwane", "Musina"],
    "Mpumalanga": ["Emalahleni (Witbank)", "Mbombela (Nelspruit)", "Kanyamazane", "Secunda", "Middelburg", "Bushbuckridge"],
    "North_West": ["Tlhabane (Rustenburg)", "Mahikeng", "Potchefstroom", "Klerksdorp", "Brits", "Jouberton"],
    "Free_State": ["Mangaung (Bloemfontein)", "Botshabelo", "Thaba 'Nchu", "Thabong (Welkom)", "Harrismith", "Sasolburg"],
    "Northern_Cape": ["Galeshewe (Kimberley)", "Paballelo (Upington)", "Kuruman", "Springbok", "De Aar", "Kathu"]
}

SA_HOSPITAL_NETWORKS = [
    "Netcare Hospital", "Life Healthcare Hospital", "Mediclinic",
    "Public Tertiary Hospital Casualty", "NHI Pilot Clinic", "District Primary Day Clinic"
]

PATIENT_DEMOGRAPHICS = [
    (f"{age}-year-old {gender}", gender)
    for age in range(18, 90)
    for gender in ["female", "male"]
]

# ==============================================================================
# STEP 3: DUAL-PLAN SCHEME RULE ENGINE & SA CONTEXT AUGMENTATION
# ==============================================================================
def assign_scheme_plan_rules_and_sa_context(row: pd.Series) -> pd.Series:
    spec = str(row["medical_specialty"]).strip()
    desc = str(row["description"]).strip()
    spec_l = spec.lower()
    desc_l = desc.lower()

    # Uniformly Sample a Scheme & Plan Pair
    scheme_key, plan_key, plan_name = random.choice(SCHEME_PLAN_REGISTRY)

    # --- RULE 1: CASH ONLY / UNINSURED ---
    if scheme_key == "Cash_Only_SelfPay":
        decision = "Out-of-Cover"
        treatment = "Direct Patient Cash Self-Pay (No Scheme Cover Applicable)"

    else:
        # --- UNIVERSAL SCHEME EXCLUSIONS (Elective / Cosmetic / Non-Formulary) ---
        out_terms = [
            "autopsy", "lipectomy", "cosmetic", "liposuction", "diet diary", "food diary",
            "supplement", "weight loss", "rhinoplasty", "breast augmentation",
            "bariatric", "tummy tuck", "botox", "chiropractic", "diets and nutritions"
        ]

        if any(t in spec_l or t in desc_l for t in out_terms):
            decision = "Out-of-Cover"
            if "autopsy" in desc_l or "autopsy" in spec_l:
                treatment = f"Post-mortem forensic autopsy examination ({scheme_key} Exclusion)"
            elif any(k in desc_l for k in ["lipectomy", "liposuction", "tummy tuck", "rhinoplasty", "augmentation"]) or "cosmetic" in spec_l:
                treatment = f"Cosmetic plastic surgery / body contouring ({scheme_key} Exclusion)"
            else:
                treatment = f"Non-covered elective/cosmetic healthcare intervention ({scheme_key} Exclusion)"

        # --- PLAN-DIFFERENTIATED POLICY RULES ---
        else:
            is_entry_network_plan = any(k in plan_key for k in ["KeyCare", "BonStart", "Ingwe", "Beat", "flexiFED"])

            if is_entry_network_plan:
                # Entry / Network Plans: Strict PMB Primary Care in-cover at network GP; all specialist/imaging/surgeries require Pre-Auth
                if any(t in desc_l for t in ["upper respiratory", "cold", "flu", "sore throat", "normal physical exam", "well child"]):
                    decision = "In-Cover"
                    treatment = f"{scheme_key} [{plan_key}] Designated Network GP Primary Care Benefit"
                elif any(t in desc_l for t in ["hypertension", "diabetes", "blood pressure", "vaginal delivery"]):
                    decision = "In-Cover"
                    treatment = f"{scheme_key} [{plan_key}] PMB Chronic / Maternal Benefit"
                else:
                    decision = "Requires Pre-Auth"
                    treatment = f"{scheme_key} [{plan_key}] Network Specialist & Inpatient Pre-Auth Required ({spec})"

            else:
                # Comprehensive / Top-Tier Plans: Broad GP & Specialist consults in-cover; inpatient surgeries & advanced imaging require Pre-Auth
                if any(t in desc_l for t in ["upper respiratory", "cold", "flu", "sore throat", "normal physical exam", "well child", "hypertension", "diabetes", "blood pressure", "vaginal delivery", "ent exam"]):
                    decision = "In-Cover"
                    treatment = f"{scheme_key} [{plan_key}] Comprehensive Primary Care Benefit"
                elif any(t in desc_l for t in ["mri", "ct scan", "surgery", "biopsy", "laparoscopy", "pacemaker", "resection", "arthroscopy"]):
                    decision = "Requires Pre-Auth"
                    treatment = f"{scheme_key} [{plan_key}] Major Inpatient / Advanced Radiology Pre-Auth Required ({spec})"
                elif spec_l in ["office notes", "general medicine", "pediatrics - neonatal", "ent - otolaryngology"]:
                    decision = "In-Cover"
                    treatment = f"{scheme_key} [{plan_key}] Comprehensive Specialist Consultation Benefit"
                else:
                    decision = "Requires Pre-Auth"
                    treatment = f"{scheme_key} [{plan_key}] Inpatient Specialist Procedural Pre-Auth ({spec})"

    # Generate SA Context Metadata
    province = random.choice(list(SA_PROVINCIAL_SUBURBS.keys()))
    suburb = random.choice(SA_PROVINCIAL_SUBURBS[province])
    demo_str, gender = random.choice(PATIENT_DEMOGRAPHICS)
    facility = random.choice(SA_HOSPITAL_NETWORKS)

    lang_key = random.choice(list(SA_GREETINGS.keys()))
    greeting = random.choice(
        SA_GREETINGS[lang_key] if lang_key == "Urban_Slang"
        else SA_GREETINGS[lang_key]["singular"] + SA_GREETINGS[lang_key]["plural"]
    )
    exclamation = random.choice(SA_EXCLAMATIONS)
    pain_str, pain_score = random.choice(SA_PAIN_DESCRIPTIONS)

    sa_header = (
        f"Claim Submission Context: {greeting}! {exclamation} "
        f"Medical Scheme & Plan Option: {plan_name}. "
        f"Facility: {facility} ({suburb}, {province.replace('_', ' ')}). "
        f"Patient Profile: {demo_str} (Reported Pain/Severity: {pain_score}/10 - {pain_str})."
    )

    return pd.Series([
        treatment, decision, scheme_key, plan_key, plan_name,
        province, suburb, facility, demo_str, gender, pain_score, sa_header
    ])

# ==============================================================================
# STEP 4: APPLY RULES & PERFORM STRATIFIED CLASS BALANCING ACROSS SCHEMES & PLANS
# ==============================================================================
print("Applying scheme & plan rules and injecting SA context metadata...")
df_raw[[
    "mock_prescription_treatment", "policy_decision", "scheme_key", "plan_key", "scheme_plan_name",
    "province", "suburb", "facility", "demographics",
    "gender", "pain_score", "sa_claim_context"
]] = df_raw.apply(assign_scheme_plan_rules_and_sa_context, axis=1)

TARGET_N = 98  # 98 samples per category = 294 total balanced samples
df_in = df_raw[df_raw["policy_decision"] == "In-Cover"].sample(n=min(TARGET_N, len(df_raw[df_raw["policy_decision"] == "In-Cover"])), random_state=42)
df_out = df_raw[df_raw["policy_decision"] == "Out-of-Cover"].sample(n=min(TARGET_N, len(df_raw[df_raw["policy_decision"] == "Out-of-Cover"])), random_state=42)
df_auth = df_raw[df_raw["policy_decision"] == "Requires Pre-Auth"].sample(n=min(TARGET_N, len(df_raw[df_raw["policy_decision"] == "Requires Pre-Auth"])), random_state=42)

df_task2_balanced = pd.concat([df_in, df_out, df_auth]).sample(frac=1, random_state=42).reset_index(drop=True)

# ==============================================================================
# STEP 5: SAVE TASK 2 OUTPUTS (JSONL)
# ==============================================================================

jsonl_output = "task2_classification_data_balanced.jsonl"
df_task2_balanced.to_json(jsonl_output, orient="records", lines=True, force_ascii=False)

# ==============================================================================
# STEP 6: SUMMARY VERIFICATION & DISTRIBUTION REPORT
# ==============================================================================
print(f"\n{'=' * 60}")
print("  TASK 2: DUAL-PLAN SCHEME ADJUDICATION DATASET REPORT")
print(f"{'=' * 60}")
print(f"Total Rows Generated: {len(df_task2_balanced)}")
print(f"Saved Files: '{jsonl_output}'\n")

print("--- 1. Policy Decision Breakdown ---")
category_counts = Counter(df_task2_balanced["policy_decision"])
for cat, count in category_counts.items():
    pct = (count / len(df_task2_balanced)) * 100
    print(f"  • {cat:45s}: {count:3d} ({pct:5.1f}%)")

print("\n--- 2. Medical Scheme Breakdown ---")
scheme_counts = Counter(df_task2_balanced["scheme_key"])
for scheme, count in sorted(scheme_counts.items()):
    pct = (count / len(df_task2_balanced)) * 100
    print(f"  • {scheme:25s}: {count:3d} ({pct:5.1f}%)")

print("\n--- 3. Scheme Plan Option Breakdown (Balanced Distribution) ---")
plan_counts = Counter(df_task2_balanced["plan_key"])
for plan, count in sorted(plan_counts.items()):
    pct = (count / len(df_task2_balanced)) * 100
    print(f"  • {plan:30s}: {count:3d} ({pct:5.1f}%)")

print("\nSample Inspection (First Example):")
print("-" * 60)
print(f"Medical Scheme & Plan    : {df_task2_balanced['scheme_plan_name'][0]}")
print(f"Policy Decision (Target) : {df_task2_balanced['policy_decision'][0]}")
print(f"Medical Specialty        : {df_task2_balanced['medical_specialty'][0]}")
print(f"Mock Treatment          : {df_task2_balanced['mock_prescription_treatment'][0]}")
print(f"SA Claim Context         :\n{textwrap.fill(str(df_task2_balanced['sa_claim_context'][0]), width=75)}")
print(f"Clinical Description     :\n{textwrap.fill(str(df_task2_balanced['description'][0]), width=75)}")
print("-" * 60)


Applying scheme & plan rules and injecting SA context metadata...

  TASK 2: DUAL-PLAN SCHEME ADJUDICATION DATASET REPORT
Total Rows Generated: 294
Saved Files: 'task2_classification_data_balanced.jsonl'

--- 1. Policy Decision Breakdown ---
  • In-Cover                                     :  98 ( 33.3%)
  • Out-of-Cover                                 :  98 ( 33.3%)
  • Requires Pre-Auth                            :  98 ( 33.3%)

--- 2. Medical Scheme Breakdown ---
  • BestMedMS                :  39 ( 13.3%)
  • BonitasMF                :  48 ( 16.3%)
  • Cash_Only_SelfPay        :  80 ( 27.2%)
  • DiscoveryHMS             :  33 ( 11.2%)
  • FedHealthMS              :  43 ( 14.6%)
  • MomentumMS               :  51 ( 17.3%)

--- 3. Scheme Plan Option Breakdown (Balanced Distribution) ---
  • Beat_Network                  :  19 (  6.5%)
  • BonComprehensive              :  24 (  8.2%)
  • BonStart_Essential            :  24 (  8.2%)
  • Executive_Comprehensive       :  21 (  7.1%)
  • 

## 5.1 Dataset inspection, cleaning and instruction tuning: Task 1

Run the following code blocks to inspect, clean and transform the conversation data into instruction tuned records.



In [ ]:
# ==============================================================================
# Task 1: Multi-Turn Conversation Data Cleaning, Stratification & Export Engine
# ==============================================================================

print("\n" + "=" * 65)
print("  STEP 1: DATA LOADING & METADATA EXTRACTION")
print("=" * 65)

# 1. Load balanced multi-turn conversation dataset
filepath = "task1_conversation_data_balanced.jsonl"
print(f"Loading dataset from: {filepath}")
df = pd.read_json(filepath, lines=True)

print(f"Loaded dataset from: {filepath}")
print(f"Total Conversations: {len(df)}")

# 2. Extract nested triage category from metadata dictionary
def extract_triage_category(meta_val: Any) -> str:
    """Extracts the policy triage category from dict or JSON string metadata."""
    if isinstance(meta_val, dict):
        return meta_val.get("triage_category") or meta_val.get("category") or "Uncategorized"
    elif isinstance(meta_val, str):
        try:
            d = json.loads(meta_val)
            return d.get("triage_category") or d.get("category") or "Uncategorized"
        except Exception:
            return "Uncategorized"
    return "Uncategorized"

# Ensure column is explicitly assigned to DataFrame
df["triage_category"] = df["meta"].apply(extract_triage_category)
print(f"Columns after extraction: {df.columns.tolist()}\n")

print("--- Initial Triage Category Distribution ---")
for cat, count in df["triage_category"].value_counts().items():
    print(f"  • {cat:45s}: {count:3d}")

print("=" * 65 + "\n")

# ==============================================================================
# STEP 2: MULTI-TURN TEXT NORMALISATION & CLEANING (EMOJI-PRESERVING)
# ==============================================================================
print("\n" + "=" * 65)
print("  STEP 2: MULTI-TURN TEXT NORMALISATION & CLEANING")
print("=" * 65)

def clean_html(text: str) -> str:
    """Strip basic HTML markup and common entities from clinical text."""
    if not isinstance(text, str):
        return ""
    text = re.sub(r"<.*?>", "", text)
    text = re.sub(r"&nbsp;", " ", text)
    text = re.sub(r"&amp;", "&", text)
    text = re.sub(r"&lt;", "<", text)
    text = re.sub(r"&gt;", ">", text)
    return text

def clean_unicode(text: str) -> str:
    """
    Removes unwanted unicode control codes while preserving:
    - L: Letters (including SA diacritics like 'môre')
    - N: Numbers
    - P: Punctuation
    - S: Symbols (preserves emergency call block emojis: 🚨, 🚑, 🚔, 🩺)
    """
    if not isinstance(text, str):
        return ""
    categories_to_keep = {"L", "N", "P", "S"}  # Preserves emergency block emojis
    keep = []
    for ch in text:
        do_keep = ch.isspace()
        if not do_keep:
            for category in categories_to_keep:
                if unicodedata.category(ch).startswith(category):
                    do_keep = True
                    break
        if do_keep:
            keep.append(ch)
    return "".join(keep)

def clean_whitespace(text: str) -> str:
    """Normalizes whitespace sequences into single spaces and strips leading/trailing spaces."""
    if not isinstance(text, str):
        return ""
    return re.sub(r"\s+", " ", text).strip()

def clean_clinical_text(text: str) -> str:
    """Applies complete normalization pipeline to a single text string."""
    return clean_whitespace(clean_unicode(clean_html(text)))

def clean_messages_list(messages: List[Dict[str, str]]) -> List[Dict[str, str]]:
    """Applies text cleaning across every turn (system, user, assistant) in a dialogue."""
    if not isinstance(messages, list):
        return []
    cleaned_messages = []
    for msg in messages:
        if isinstance(msg, dict) and "content" in msg:
            cleaned_msg = msg.copy()
            cleaned_msg["content"] = clean_clinical_text(msg["content"])
            cleaned_messages.append(cleaned_msg)
        else:
            cleaned_messages.append(msg)
    return cleaned_messages

# Apply multi-turn cleaning across dialogue messages
if "messages" in df.columns:
    df["messages"] = df["messages"].apply(clean_messages_list)

print("✅ Applied emoji-preserving text normalisation across all multi-turn dialogue turns.")
print("=" * 65 + "\n")

# ==============================================================================
# STEP 3: DYNAMIC STRATIFIED PARTITIONING (80% TRAIN / 10% VAL / 10% TEST)
# ==============================================================================
print("\n" + "=" * 65)
print("  STEP 3: DYNAMIC STRATIFIED DATASET PARTITIONING")
print("=" * 65)

# Shuffle dataset with fixed seed for reproducibility
df_shuffled = df.sample(frac=1, random_state=42).reset_index(drop=True)

# 1. Stratified Split into 80% Train (192) and 20% Unseen Pool (48)
df_train, df_unseen = train_test_split(
    df_shuffled,
    test_size=0.20,
    random_state=42,
    stratify=df_shuffled["triage_category"],
)

# 2. Stratified 50/50 Split of Unseen Pool into Validation (24) and Test (24)
df_val, df_test = train_test_split(
    df_unseen,
    test_size=0.50,
    random_state=42,
    stratify=df_unseen["triage_category"],
)

# Reset indices
df_train = df_train.reset_index(drop=True)
df_val = df_val.reset_index(drop=True)
df_test = df_test.reset_index(drop=True)

print(f"✅ Partitioning complete: Train ({len(df_train)}), Val ({len(df_val)}), Test ({len(df_test)}).")
print("=" * 65 + "\n")

# ==============================================================================
# STEP 4: PARTITION VERIFICATION & DISTRIBUTION ANALYTICS
# ==============================================================================
print("\n" + "=" * 65)
print("  STEP 4: PARTITION VERIFICATION & DISTRIBUTION ANALYTICS")
print("=" * 65)

print(f"--- 1. Training Set Breakdown ({len(df_train)} total conversations) ---")
for cat, count in df_train["triage_category"].value_counts().items():
    pct = (count / len(df_train)) * 100
    print(f"  • {cat:45s}: {count:3d} ({pct:5.1f}%)")

print(f"\n--- 2. Validation Set Breakdown ({len(df_val)} total conversations) ---")
for cat, count in df_val["triage_category"].value_counts().items():
    pct = (count / len(df_val)) * 100
    print(f"  • {cat:45s}: {count:3d} ({pct:5.1f}%)")

print(f"\n--- 3. Test Set Breakdown ({len(df_test)} total conversations) ---")
for cat, count in df_test["triage_category"].value_counts().items():
    pct = (count / len(df_test)) * 100
    print(f"  • {cat:45s}: {count:3d} ({pct:5.1f}%)")

print("=" * 65 + "\n")

# ==============================================================================
# STEP 5: MULTI-TURN INSTRUCTION EXPORT TO JSONL
# ==============================================================================
print("\n" + "=" * 65)
print("  STEP 5: INSTRUCTION FORMAT EXPORT TO JSONL")
print("=" * 65)

def format_record_for_export(row: pd.Series) -> Dict[str, Any]:
    """Formats row into multi-turn schema or input/output pair."""
    if "messages" in row and isinstance(row["messages"], list):
        return {
            "case_id": row.get("case_id", ""),
            "messages": row["messages"],
            "meta": row.get("meta", {})
        }
    else:
        return {
            "input": str(row.get("prompt", "")),
            "output": str(row.get("response", ""))
        }

export_train = [format_record_for_export(r) for _, r in df_train.iterrows()]
export_val = [format_record_for_export(r) for _, r in df_val.iterrows()]
export_test = [format_record_for_export(r) for _, r in df_test.iterrows()]

print("--- Sample Multi-Turn Record Preview ---")
print(json.dumps(export_train[0], ensure_ascii=False, indent=2))
print("-" * 65 + "\n")

files_to_export = [
    ("conversation_training_data.jsonl", export_train),
    ("conversation_validation_data.jsonl", export_val),
    ("conversation_test_data.jsonl", export_test),
]

for filename, records in files_to_export:
    with open(filename, "w", encoding="utf-8") as f:
        for rec in records:
            f.write(json.dumps(rec, ensure_ascii=False) + "\n")
    print(f"📁 Successfully exported {len(records):3d} conversation records to '{filename}'")

print("\n" + "=" * 65)
print("  ✅ TASK 1 MULTI-TURN DATA CLEANING & PARTITIONING COMPLETE")
print("=" * 65 + "\n")


  STEP 1: DATA LOADING & METADATA EXTRACTION
Loading dataset from: task1_conversation_data_balanced.jsonl
Loaded dataset from: task1_conversation_data_balanced.jsonl
Total Conversations: 140
Columns after extraction: ['case_id', 'messages', 'translated_user_turns', 'meta', 'triage_category']

--- Initial Triage Category Distribution ---
  • Requires Pre-Auth / Emergency Triage         :  80
  • Out-of-Cover (Scheme Exclusion)              :  38
  • In-Cover (Primary Care / GP Consultation)    :  22


  STEP 2: MULTI-TURN TEXT NORMALISATION & CLEANING
✅ Applied emoji-preserving text normalisation across all multi-turn dialogue turns.


  STEP 3: DYNAMIC STRATIFIED DATASET PARTITIONING
✅ Partitioning complete: Train (112), Val (14), Test (14).


  STEP 4: PARTITION VERIFICATION & DISTRIBUTION ANALYTICS
--- 1. Training Set Breakdown (112 total conversations) ---
  • Requires Pre-Auth / Emergency Triage         :  64 ( 57.1%)
  • Out-of-Cover (Scheme Exclusion)              :  30 ( 26.8%)

## 5.2 Dataset inspection, cleaning and instruction tuning: Task 2

Run the following code blocks to inspect, clean and transform the classification data into instruction tuned records.



In [ ]:
# ==============================================================================
# STEP 1: LOAD BALANCED MASTER DATASET
# ==============================================================================
print("=" * 65)
print("  STEP 1: LOADING MASTER CLASSIFICATION DATASET")
print("=" * 65)

input_file = "task2_classification_data_balanced.jsonl"
if not os.path.exists(input_file):
    input_file = "task2_classification_data.csv"

if input_file.endswith(".csv"):
    df_raw = pd.read_csv(input_file)
else:
    df_raw = pd.read_json(input_file, lines=True)

print(f"✅ Loaded {len(df_raw)} total balanced claim records.")

# ==============================================================================
# STEP 2: INSTRUCTION FORMATTING FUNCTION (to_instruction_record)
# ==============================================================================
def to_instruction_record(row: pd.Series) -> dict[str, str]:
    """
    Transforms a raw medical claim row into a clean instruction-tuning record.
    Strips conversational noise and keeps context structured & concise.
    """
    instruction = (
        "Classify the following South African medical scheme claim submission "
        "into one of the policy decision categories: 'In-Cover', 'Out-of-Cover', or 'Requires Pre-Auth'.\n\n"
    )

    # Extract clean fields with graceful fallbacks
    scheme_plan = row.get("scheme_plan_name", row.get("plan_name", "N/A"))
    facility = row.get("facility", "N/A")
    province = str(row.get("province", "N/A")).replace("_", " ")
    suburb = row.get("suburb", "N/A")
    specialty = row.get("medical_specialty", "N/A")
    description = row.get("description", "N/A")

    # Structured context without conversational greetings
    context = (
        f"Medical Scheme & Plan: {scheme_plan}\n"
        f"Facility: {facility} ({suburb}, {province})\n"
        f"Medical Specialty: {specialty}\n"
        f"Clinical Description: {description}"
    )

    return {
        "input": instruction + context,
        "output": str(row["policy_decision"]).strip()
    }

# ==============================================================================
# STEP 3: DYNAMIC STRATIFIED DATASET PARTITIONING (80 / 10 / 10)
# ==============================================================================
print("\n" + "=" * 65)
print("  STEP 3: STRATIFIED SPLITTING (80% Train / 10% Val / 10% Test)")
print("=" * 65)

# 80% Train, 20% Temp (Val + Test)
df_train, df_temp = train_test_split(
    df_raw,
    test_size=0.20,
    random_state=812,
    stratify=df_raw["policy_decision"]
)

# Split 20% Temp evenly into 10% Validation and 10% Test
df_val, df_test = train_test_split(
    df_temp,
    test_size=0.50,
    random_state=812,
    stratify=df_temp["policy_decision"]
)

print(f"✅ Partitioning Complete:")
print(f"  • Training   : {len(df_train)} claims (80%)")
print(f"  • Validation : {len(df_val)} claims (10%)")
print(f"  • Testing    : {len(df_test)} claims (10%)")

# ==============================================================================
# STEP 4: TRANSFORM SPLITS TO INSTRUCTION RECORDS & EXPORT JSONL
# ==============================================================================
print("\n" + "=" * 65)
print("  STEP 4: TRANSFORMING RECORDS & EXPORTING JSONL PARTITIONS")
print("=" * 65)

instruction_records_train = [to_instruction_record(r) for _, r in df_train.iterrows()]
instruction_records_val = [to_instruction_record(r) for _, r in df_val.iterrows()]
instruction_records_test = [to_instruction_record(r) for _, r in df_test.iterrows()]

# Preview a single record
print("--- Sample Instruction Record Preview ---")
print(json.dumps(instruction_records_train[0], ensure_ascii=False, indent=2))
print("-" * 65)

# Save JSONL partitions
splits = [
    ("classification_training_data.jsonl", instruction_records_train),
    ("classification_validation_data.jsonl", instruction_records_val),
    ("classification_test_data.jsonl", instruction_records_test),
]

for filepath, records in splits:
    with open(filepath, "w", encoding="utf-8") as f:
        for rec in records:
            f.write(json.dumps(rec, ensure_ascii=False) + "\n")
    print(f"✅ Saved {len(records)} records to '{filepath}'")

print("\n✅ Task 2 Data Cleaning & Partition Export Complete!")
print("=" * 65 + "\n")

  STEP 1: LOADING MASTER CLASSIFICATION DATASET
✅ Loaded 294 total balanced claim records.

  STEP 3: STRATIFIED SPLITTING (80% Train / 10% Val / 10% Test)
✅ Partitioning Complete:
  • Training   : 235 claims (80%)
  • Validation : 29 claims (10%)
  • Testing    : 30 claims (10%)

  STEP 4: TRANSFORMING RECORDS & EXPORTING JSONL PARTITIONS
--- Sample Instruction Record Preview ---
{
  "input": "Classify the following South African medical scheme claim submission into one of the policy decision categories: 'In-Cover', 'Out-of-Cover', or 'Requires Pre-Auth'.\n\nMedical Scheme & Plan: Bestmed Pace Series (Comprehensive Plan)\nFacility: NHI Pilot Clinic (Khayelitsha, Western Cape)\nMedical Specialty: Nephrology\nClinical Description: Patient with end-stage renal disease secondary to hypertension, a reasonable candidate for a kidney transplantation.",
  "output": "In-Cover"
}
-----------------------------------------------------------------
✅ Saved 235 records to 'classification_training_da

## 6.1 Training and evaluation: Task 1

Run the following code blocks to fine tune a pre-trained Gemma 3-1B module and evaluate your model performance on the cleaned conversation data.



In [ ]:
# ==============================================================================
# STEP 0: DATA LOADING & RECORD INSPECTION (TASK 1: CONVERSATIONAL TRIAGE)
# ==============================================================================
import pandas as pd

print("\n" + "=" * 65)
print("  STEP 0: DATA LOADING & RECORD INSPECTION (TASK 1)")
print("=" * 65)

# 1. Load Training and Validation Data
train_df = pd.read_json("conversation_training_data.jsonl", lines=True)
val_df = pd.read_json("conversation_validation_data.jsonl", lines=True)

# 2. Load Test Data
test_conversation_df = pd.read_json("conversation_test_data.jsonl", lines=True)

# 3. Convert DataFrames into lists of dictionaries expected by fine-tuning code
train_conversation_data = train_df.to_dict("records")
val_conversation_data = val_df.to_dict("records")
test_conversation_data = test_conversation_df.to_dict("records")

# 4. Print dataset sizes and first records for verification
print(f"Conversational Training Records   : {len(train_conversation_data)}")
print(f"Conversational Validation Records : {len(val_conversation_data)}")
print(f"Conversational Test Records       : {len(test_conversation_data)}\n")

print("--- Sample Train Record ---")
print(train_conversation_data[0])
print("\n--- Sample Validation Record ---")
print(val_conversation_data[0])
print("\n--- Sample Conversation Test Record ---")
print(test_conversation_data[0])
print("=" * 65 + "\n")


  STEP 0: DATA LOADING & RECORD INSPECTION (TASK 1)
Conversational Training Records   : 112
Conversational Validation Records : 14
Conversational Test Records       : 14

--- Sample Train Record ---
{'case_id': 'SA-CASE-0060', 'messages': [{'role': 'system', 'content': 'You are Care2ClaimLLM-ZA, an empathetic South African AI healthcare and coverage decision guide. Your mission is to support South Africans experiencing medical distress and health anxiety with clarity and dignity. Initiate every interaction with a warm, supportive check-in to hear how the patient is doing. Gently gather essential intake details—age, gender, medical scheme and plan option (or cash/self-pay), approximate location (suburb and province), and pain level on a 1 to 10 scale—to provide tailored primary care triage and scheme coverage guidance without making unauthorized clinical diagnoses.'}, {'role': 'user', 'content': "Thobelang! Lekker man How's the body? I'm writing from Mangaung (Bloemfontein) in Free Sta

In [ ]:
# ==============================================================================
# STEP 1: GEMMA 3 CONTROL TOKENS & MULTI-TURN DIALOGUE FORMATTER
# ==============================================================================
print("\n" + "=" * 65)
print("  STEP 1: GEMMA 3 CONTROL TOKENS & MULTI-TURN DIALOGUE FORMATTER")
print("=" * 65)

SOT = "<start_of_turn>"
EOT = "<end_of_turn>"

def extract_dialogue_pair(record: dict) -> Tuple[str, str]:
    """
    Extracts formatted input prompt history (Turns 1 to 3) and target
    assistant administrative triage guidance (Turn 4) from a Task 1 conversation record.
    """
    if "messages" in record and isinstance(record["messages"], list):
        messages = record["messages"]
        history_parts = []
        # Format Turns 1 through N-1 as dialogue history
        for msg in messages[:-1]:
            role = "user" if msg.get("role") in ["user", "system"] else "model"
            content = msg.get("content", "")
            history_parts.append(f"{SOT}{role}\n{content}{EOT}")
        prompt_history = "\n".join(history_parts)
        target_response = messages[-1].get("content", "")
        return prompt_history, target_response
    elif "input" in record and "output" in record:
        return f"{SOT}user\n{record['input']}{EOT}", str(record["output"])
    elif "prompt" in record and "response" in record:
        return f"{SOT}user\n{record['prompt']}{EOT}", str(record["response"])
    else:
        return "", ""

def format_io_as_turns(input_prompt: str, target_output: str) -> Tuple[str, str]:
    """
    Formats the dialogue history and target output into KerasNLP user/model turns.
    """
    formatted_q = input_prompt.strip() + "\n" if not input_prompt.endswith("\n") else input_prompt
    formatted_a = f"{SOT}model\n{target_output}{EOT}" if not target_output.startswith(SOT) else target_output
    return formatted_q, formatted_a

# ==============================================================================
# STEP 2: LOAD JSONL PARTITIONS & PREPARE CONVERSATIONAL DATASETS
# ==============================================================================
print("\n" + "=" * 65)
print("  STEP 2: LOADING PREPROCESSED CONVERSATION PARTITIONS")
print("=" * 65)

def load_partition_records(filename: str) -> Tuple[List[Dict[str, Any]], List[str], List[str]]:
    """Loads clean JSONL conversation records and returns raw dicts, formatted prompts, and target responses."""
    if not os.path.exists(filename):
        alt_filename = filename.replace("conversation_", "task1_conversation_")
        if os.path.exists(alt_filename):
            filename = alt_filename
        elif os.path.exists("task1_conversation_data_balanced.jsonl"):
            filename = "task1_conversation_data_balanced.jsonl"
        else:
            raise FileNotFoundError(f"Missing required partition file: '{filename}'")

    raw_records = []
    inputs = []
    targets = []

    with open(filename, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                rec = json.loads(line.strip())
                in_text, out_text = extract_dialogue_pair(rec)
                q, a = format_io_as_turns(in_text, out_text)
                raw_records.append(rec)
                inputs.append(q)
                targets.append(a)

    return raw_records, inputs, targets

# Load dataset splits (Task 1 Conversation Partitions)
train_records, train_inputs, train_targets = load_partition_records("conversation_training_data.jsonl")
val_records, val_inputs, val_targets = load_partition_records("conversation_validation_data.jsonl")
test_records, test_inputs, test_targets = load_partition_records("conversation_test_data.jsonl")

# Format dictionary inputs for Gemma 3 Causal LM Preprocessor
train_dataset = {"prompts": train_inputs, "responses": train_targets}
val_dataset = {"prompts": val_inputs, "responses": val_targets}

print(f"✅ Conversational Data Partitions Loaded & Formatted for KerasNLP:")
print(f"  • Training   : {len(train_inputs)} multi-turn dialogue pairs")
print(f"  • Validation : {len(val_inputs)} multi-turn dialogue pairs")
print(f"  • Testing    : {len(test_inputs)} multi-turn dialogue pairs\n")

print("--- Sample Multi-Turn Prompt History Preview ---")
print(train_inputs[0][:250] + "...\n")
print("--- Sample Target Assistant Response Preview ---")
print(train_targets[0][:200] + "...\n")

# ==============================================================================
# STEP 3: GEMMA 3 1B MODEL LOADING & PRECISION CONFIGURATION
# ==============================================================================
print("=" * 65)
print("  STEP 3: GEMMA 3 1B MODEL LOADING & MEMORY CONFIGURATION")
print("=" * 65)

# 1. Enable mixed_bfloat16 precision policy for memory efficiency on NVIDIA T4 GPU
keras.config.set_dtype_policy("mixed_bfloat16")
keras.utils.set_random_seed(812)

# 2. Load Gemma 3 1B base preset
preset_name = "gemma3_1b"
print(f"Loading preset '{preset_name}' in mixed_bfloat16 precision...")
model = keras_nlp.models.Gemma3CausalLM.from_preset(preset_name)

# 3. Set sequence length ceiling to 512 tokens to handle multi-turn intake dialogue
if hasattr(model, "preprocessor") and model.preprocessor is not None:
    model.preprocessor.sequence_length = 512
    print("✅ Preprocessor sequence length set to 512 tokens.")

print("=" * 65 + "\n")


  STEP 1: GEMMA 3 CONTROL TOKENS & MULTI-TURN DIALOGUE FORMATTER

  STEP 2: LOADING PREPROCESSED CONVERSATION PARTITIONS
✅ Conversational Data Partitions Loaded & Formatted for KerasNLP:
  • Training   : 112 multi-turn dialogue pairs
  • Validation : 14 multi-turn dialogue pairs
  • Testing    : 14 multi-turn dialogue pairs

--- Sample Multi-Turn Prompt History Preview ---
<start_of_turn>user
You are Care2ClaimLLM-ZA, an empathetic South African AI healthcare and coverage decision guide. Your mission is to support South Africans experiencing medical distress and health anxiety with clarity and dignity. Initiate every i...

--- Sample Target Assistant Response Preview ---
<start_of_turn>model
Kua right / Keep safe! If you decide to visit your local retail pharmacy clinic or private GP (Self-Pay), remember to take your ID and FedHealthMS card if applicable. Please reach...

  STEP 3: GEMMA 3 1B MODEL LOADING & MEMORY CONFIGURATION
Loading preset 'gemma3_1b' in mixed_bfloat16 precision..

In [ ]:
# ==============================================================================
# STEP 4: PARAMETER-EFFICIENT FINE-TUNING (LoRA) & TRAINING LOOP
# ==============================================================================
print("=" * 65)
print("  STEP 4: LoRA FINE-TUNING SETUP & MULTI-TURN TRAINING")
print("=" * 65)

# 1. Enable LoRA rank=8 backbone adaptation (~0.1% trainable parameters)
lora_rank = 8
if hasattr(model.backbone, "lora_enabled") and model.backbone.lora_enabled:
    print(f"ℹ️ LoRA adapter (rank={lora_rank}) is already enabled on Gemma 3 backbone.")
else:
    try:
        model.backbone.enable_lora(rank=lora_rank)
        print(f"✅ Enabled LoRA on Gemma 3 backbone (rank={lora_rank}).")
    except (ValueError, AttributeError) as e:
        print(f"ℹ️ LoRA adapter already active on backbone: {e}")

model.summary()

# 2. Compile model with AdamW optimizer & XLA acceleration
model.compile(
    optimizer=keras.optimizers.AdamW(learning_rate=1e-4, weight_decay=0.01),
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=[keras.metrics.SparseCategoricalAccuracy(name="accuracy")],
    jit_compile=True
)

print("\nStarting LoRA Multi-Turn Fine-Tuning Execution (batch_size=1)...")
epochs = 3
batch_size = 1  # batch_size=1 preserves VRAM headroom under 512 token sequence length

history = model.fit(
    x=train_dataset,
    epochs=epochs,
    batch_size=batch_size
)

print("\n✅ Task 1 LoRA Fine-Tuning Complete!")
print("=" * 65 + "\n")

  STEP 4: LoRA FINE-TUNING SETUP & MULTI-TURN TRAINING
✅ Enabled LoRA on Gemma 3 backbone (rank=8).


Preprocessor: "gemma3_causal_lm_preprocessor"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                                                  ┃                                   Config ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ gemma3_tokenizer (Gemma3Tokenizer)                            │                      Vocab size: 262,144 │
└───────────────────────────────────────────────────────────────┴──────────────────────────────────────────┘

Model: "gemma3_causal_lm"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                  ┃ Output Shape              ┃         Param # ┃ Connected to               ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ padding_mask (InputLayer)     │ (None, None)              │               0 │ -                          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ token_ids (InputLayer)        │ (None, None)              │               0 │ -                          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ gemma3_backbone               │ (None, None, 1152)        │   1,001,190,528 │ padding_mask[0][0],        │
│ (Gemma3Backbone)              │                           │                 │ token_ids[0][0]            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ token_embedding               │ (None, None, 262144)      │     301,989,888 │ gemma3_backbone[0][0]      │
│ (ReversibleEmbedding)         │                           │                 │                            │
└───────────────────────────────┴───────────────────────────┴─────────────────┴────────────────────────────┘

 Total params: 1,001,190,528 (3.73 GB)

 Trainable params: 1,304,576 (4.98 MB)

 Non-trainable params: 999,885,952 (3.72 GB)


Starting LoRA Multi-Turn Fine-Tuning Execution (batch_size=1)...
Epoch 1/3
112/112 ━━━━━━━━━━━━━━━━━━━━ 92s 528ms/step - accuracy: 0.4790 - loss: 0.3311 - sparse_categorical_accuracy: 0.5355
Epoch 2/3
112/112 ━━━━━━━━━━━━━━━━━━━━ 64s 568ms/step - accuracy: 0.4004 - loss: 0.0327 - sparse_categorical_accuracy: 0.9642
Epoch 3/3
112/112 ━━━━━━━━━━━━━━━━━━━━ 67s 590ms/step - accuracy: 0.3512 - loss: 0.0013 - sparse_categorical_accuracy: 0.9969

✅ Task 1 LoRA Fine-Tuning Complete!



In [ ]:
# ==============================================================================
# STEP 5: FAST BATCHED PREDICTION GENERATION ACROSS SPLITS
# ==============================================================================
print("=" * 65)
print("  STEP 5: FAST BATCHED PREDICTION GENERATION ACROSS SPLITS")
print("=" * 65)

def generate_predictions_batched(
    records: list,
    model: keras_nlp.models.Gemma3CausalLM,
    output_jsonl_file: str,
    batch_size: int = 8,
    max_samples: int = None,
) -> pd.DataFrame:
    """Generates dialogue predictions in GPU vectorized batches."""
    if max_samples and len(records) > max_samples:
        random.seed(812)
        records_to_process = random.sample(records, k=max_samples)
        print(f"Sampling {max_samples} records from {len(records)} total for fast evaluation.")
    else:
        records_to_process = records

    df_pred = pd.DataFrame(records_to_process)
    df_pred["model_output"] = ""

    # Build inference prompts ending with assistant turn header
    inference_prompts = []
    for rec in records_to_process:
        in_text, _ = extract_dialogue_pair(rec)
        q, _ = format_io_as_turns(in_text, "")
        prompt_full = f"{q}{SOT}model\n"
        inference_prompts.append(prompt_full)

    print(f"Running batched generation ({len(df_pred)} total records, batch_size={batch_size})...")

    generated_outputs = []
    for i in tqdm(range(0, len(inference_prompts), batch_size)):
        batch_prompts = inference_prompts[i : i + batch_size]
        batch_results = model.generate(batch_prompts, max_length=512)
        generated_outputs.extend(batch_results)

    df_pred["model_output"] = generated_outputs
    df_pred.to_json(output_jsonl_file, orient="records", lines=True, force_ascii=False)
    print(f"✅ Saved predictions to '{output_jsonl_file}'.")
    return df_pred

# 1. Sample 30 Training Records for Seen Evaluation
df_pred_train = generate_predictions_batched(
    train_records, model, "predictions_conversation_train.jsonl", batch_size=8, max_samples=30
)

# 2. Complete Validation Set (24 records)
df_pred_val = generate_predictions_batched(
    val_records, model, "predictions_conversation_val.jsonl", batch_size=8
)

# 3. Complete Test Set (24 records)
df_pred_test = generate_predictions_batched(
    test_records, model, "predictions_conversation_test.jsonl", batch_size=8
)


  STEP 5: FAST BATCHED PREDICTION GENERATION ACROSS SPLITS
Sampling 30 records from 112 total for fast evaluation.
Running batched generation (30 total records, batch_size=8)...


100%|██████████| 4/4 [01:11<00:00, 17.86s/it]


✅ Saved predictions to 'predictions_conversation_train.jsonl'.
Running batched generation (14 total records, batch_size=8)...


100%|██████████| 2/2 [00:02<00:00,  1.45s/it]


✅ Saved predictions to 'predictions_conversation_val.jsonl'.
Running batched generation (14 total records, batch_size=8)...


100%|██████████| 2/2 [00:02<00:00,  1.46s/it]

✅ Saved predictions to 'predictions_conversation_test.jsonl'.


In [ ]:
# ==============================================================================
# STEP 6: NATURAL LANGUAGE GENERATION (NLG) EVALUATION METRICS
# ==============================================================================
print("\n" + "=" * 65)
print("  STEP 6: NATURAL LANGUAGE GENERATION (NLG) EVALUATION METRICS")
print("=" * 65)

# Safely check if SentenceTransformer is available in the current environment
try:
    from sentence_transformers import SentenceTransformer
    HAS_SENTENCE_TRANSFORMERS = True
except ImportError:
    HAS_SENTENCE_TRANSFORMERS = False

# Initialize ROUGE Scorer & SentenceTransformer / TF-IDF
r_scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)
smooth_fn = SmoothingFunction().method1

if HAS_SENTENCE_TRANSFORMERS:
    embedder = SentenceTransformer('all-MiniLM-L6-v2')
else:
    embedder = None
    print("ℹ️ sentence-transformers not installed; falling back to TF-IDF cosine similarity.")

def clean_generated_text(text: str) -> str:
    """Extracts raw assistant response text from generated turn string."""
    if not isinstance(text, str):
        return ""
    if f"{SOT}model" in text:
        text = text.rsplit(f"{SOT}model", 1)[-1]
    text = text.replace(EOT, "").strip()
    return text

def extract_target_text(rec: dict) -> str:
    _, target = extract_dialogue_pair(rec)
    return target

def evaluate_nlg_performance(df_input: pd.DataFrame, title: str) -> dict:
    """Computes ROUGE-1, ROUGE-2, ROUGE-L, BLEU-2, and Semantic Cosine Similarity."""
    df_eval = df_input.copy()

    references = [extract_target_text(row.to_dict()) for _, row in df_eval.iterrows()]
    hypotheses = [clean_generated_text(row["model_output"]) for _, row in df_eval.iterrows()]

    r1_scores, r2_scores, rl_scores = [], [], []
    bleu_scores = []

    for ref, hyp in zip(references, hypotheses):
        # ROUGE
        scores = r_scorer.score(ref, hyp)
        r1_scores.append(scores['rouge1'].fmeasure)
        r2_scores.append(scores['rouge2'].fmeasure)
        rl_scores.append(scores['rougeL'].fmeasure)

        # BLEU-2
        ref_tokens = nltk.word_tokenize(ref.lower())
        hyp_tokens = nltk.word_tokenize(hyp.lower())
        bleu = sentence_bleu([ref_tokens], hyp_tokens, weights=(0.5, 0.5, 0, 0), smoothing_function=smooth_fn)
        bleu_scores.append(bleu)

    # Semantic Cosine Similarity
    if embedder is not None:
        ref_emb = embedder.encode(references)
        hyp_emb = embedder.encode(hypotheses)
        sims = np.diag(cosine_similarity(ref_emb, hyp_emb))
    else:
        vectorizer = TfidfVectorizer().fit(references + hypotheses)
        sims = [cosine_similarity(vectorizer.transform([r]), vectorizer.transform([h]))[0][0]
                for r, h in zip(references, hypotheses)]

    mean_r1 = np.mean(r1_scores)
    mean_r2 = np.mean(r2_scores)
    mean_rl = np.mean(rl_scores)
    mean_bleu = np.mean(bleu_scores)
    mean_sim = np.mean(sims)

    print(f"\n--- {title} ---")
    print(f"ROUGE-1 (Unigram Overlap)   : {mean_r1:.4f}")
    print(f"ROUGE-2 (Bigram Overlap)    : {mean_r2:.4f}")
    print(f"ROUGE-L (Longest Sequence)  : {mean_rl:.4f}")
    print(f"BLEU-2 (Precision Metric)   : {mean_bleu:.4f}")
    print(f"Semantic Cosine Similarity  : {mean_sim:.4f}")

    return {
        "rouge1": mean_r1,
        "rouge2": mean_r2,
        "rougeL": mean_rl,
        "bleu2": mean_bleu,
        "semantic_sim": mean_sim,
    }

results_train = evaluate_nlg_performance(df_pred_train, "Training Partition NLG Performance")
results_val = evaluate_nlg_performance(df_pred_val, "Validation Partition NLG Performance")
results_test = evaluate_nlg_performance(df_pred_test, "Test Partition NLG Performance")



  STEP 6: NATURAL LANGUAGE GENERATION (NLG) EVALUATION METRICS


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

... (5 outputs left out)

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]


--- Training Partition NLG Performance ---
ROUGE-1 (Unigram Overlap)   : 0.2131
ROUGE-2 (Bigram Overlap)    : 0.0276
ROUGE-L (Longest Sequence)  : 0.1479
BLEU-2 (Precision Metric)   : 0.0306
Semantic Cosine Similarity  : 0.4332

--- Validation Partition NLG Performance ---
ROUGE-1 (Unigram Overlap)   : 0.2291
ROUGE-2 (Bigram Overlap)    : 0.0552
ROUGE-L (Longest Sequence)  : 0.1497
BLEU-2 (Precision Metric)   : 0.0465
Semantic Cosine Similarity  : 0.4435

--- Test Partition NLG Performance ---
ROUGE-1 (Unigram Overlap)   : 0.2159
ROUGE-2 (Bigram Overlap)    : 0.0410
ROUGE-L (Longest Sequence)  : 0.1474
BLEU-2 (Precision Metric)   : 0.0378
Semantic Cosine Similarity  : 0.4461


In [ ]:
# ==============================================================================
# STEP 7: PROMPT SENSITIVITY & ROBUSTNESS ANALYSIS
# ==============================================================================
print("\n" + "=" * 65)
print("  STEP 7: PROMPT SENSITIVITY & ROBUSTNESS ANALYSIS")
print("=" * 65)

# Evaluate model robustness across South African greeting variations & phrasing
sensitivity_test_cases = [
    {
        "variant": "isiZulu Greeting ('Sawubona')",
        "prompt": f"{SOT}user\nSawubona dokotela! I have a severe burning chest pain when breathing.\n{EOT}\n{SOT}model\nSawubona! Please tell me your medical scheme and plan option.\n{EOT}\n{SOT}user\nI am on Discovery KeyCare Access at Netcare Park Lane.\n{EOT}\n{SOT}model\n"
    },
    {
        "variant": "isiXhosa Greeting ('Molo dokotela')",
        "prompt": f"{SOT}user\nMolo dokotela! I have a severe burning chest pain when breathing.\n{EOT}\n{SOT}model\nMolo! Please tell me your medical scheme and plan option.\n{EOT}\n{SOT}user\nI am on Discovery KeyCare Access at Netcare Park Lane.\n{EOT}\n{SOT}model\n"
    },
    {
        "variant": "Urban Slang Greeting ('Aweh doc / Sho')",
        "prompt": f"{SOT}user\nAweh doc! Yoh, sharp pain in my chest when I cough, sho!\n{EOT}\n{SOT}model\nHowzit! Please let me know your medical scheme and plan.\n{EOT}\n{SOT}user\nI am on Discovery KeyCare Access at Netcare Park Lane.\n{EOT}\n{SOT}model\n"
    },
    {
        "variant": "English Formal Greeting ('Good day doctor')",
        "prompt": f"{SOT}user\nGood day doctor, I am experiencing acute chest discomfort.\n{EOT}\n{SOT}model\nGood day! What is your medical scheme and current facility?\n{EOT}\n{SOT}user\nI am on Discovery KeyCare Access at Netcare Park Lane.\n{EOT}\n{SOT}model\n"
    }
]

print("Running Prompt Sensitivity & Dialect Robustness Evaluation...\n")
sens_prompts = [c["prompt"] for c in sensitivity_test_cases]
sens_results = model.generate(sens_prompts, max_length=512)

for case, raw_output in zip(sensitivity_test_cases, sens_results):
    clean_resp = clean_generated_text(raw_output)
    print(f"--- Variant: {case['variant']} ---")
    print(f"Generated Response:\n{clean_resp[:180]}...\n")


  STEP 7: PROMPT SENSITIVITY & ROBUSTNESS ANALYSIS
Running Prompt Sensitivity & Dialect Robustness Evaluation...

--- Variant: isiZulu Greeting ('Sawubona') ---
Generated Response:
Please guide me if necessary....

--- Variant: isiXhosa Greeting ('Molo dokotela') ---
Generated Response:
Molo! Take me to the nearest NetCare hospital emergency....

--- Variant: Urban Slang Greeting ('Aweh doc / Sho') ---
Generated Response:
Keep me updated if your symptoms improve!...

--- Variant: English Formal Greeting ('Good day doctor') ---
Generated Response:
Let me check and provide further advice if necessary....



In [ ]:
# ==============================================================================
# STEP 8: TASK 1 SUMMARY PERFORMANCE COMPARISON REPORT
# ==============================================================================
print("\n" + "=" * 65)
print("  TASK 1 SUMMARY: MULTI-TURN CONVERSATIONAL TRIAGE BENCHMARK")
print("=" * 65)
print(f"{'Metric':<25} {'Train (Seen)':<15} {'Validation':<15} {'Test (Unseen)':<15}")
print("-" * 65)
print(f"{'ROUGE-1 (Unigram)':<25} {results_train['rouge1']:.4f}{'':<10} {results_val['rouge1']:.4f}{'':<10} {results_test['rouge1']:.4f}")
print(f"{'ROUGE-2 (Bigram)':<25} {results_train['rouge2']:.4f}{'':<10} {results_val['rouge2']:.4f}{'':<10} {results_test['rouge2']:.4f}")
print(f"{'ROUGE-L (LCS)':<25} {results_train['rougeL']:.4f}{'':<10} {results_val['rougeL']:.4f}{'':<10} {results_test['rougeL']:.4f}")
print(f"{'BLEU-2 (Precision)':<25} {results_train['bleu2']:.4f}{'':<10} {results_val['bleu2']:.4f}{'':<10} {results_test['bleu2']:.4f}")
print(f"{'Semantic Similarity':<25} {results_train['semantic_sim']:.4f}{'':<10} {results_val['semantic_sim']:.4f}{'':<10} {results_test['semantic_sim']:.4f}")
print("=" * 65 + "\n")


  TASK 1 SUMMARY: MULTI-TURN CONVERSATIONAL TRIAGE BENCHMARK
Metric                    Train (Seen)    Validation      Test (Unseen)  
-----------------------------------------------------------------
ROUGE-1 (Unigram)         0.2131           0.2291           0.2159
ROUGE-2 (Bigram)          0.0276           0.0552           0.0410
ROUGE-L (LCS)             0.1479           0.1497           0.1474
BLEU-2 (Precision)        0.0306           0.0465           0.0378
Semantic Similarity       0.4332           0.4435           0.4461



# Free Memory Between Tasks

Run the following code block to reset the weights and Gemma3-1B model for the next task.


In [ ]:
import gc
if "model" in locals():
    del model
if "history" in locals():
    del history

gc.collect()
keras.backend.clear_session()

print("✅ Cleared Keras session & freed GPU VRAM for Task 2!")

✅ Cleared Keras session & freed GPU VRAM for Task 2!


## 6.2 Training and evaluation: Task 2

Run the following code blocks to fine tune a pre-trained Gemma 3-1B module and evaluate your model performance on the cleaned classification data.



In [ ]:
# ==============================================================================
# STEP 0: DATA LOADING & RECORD INSPECTION (TASK 2: CLAIM CLASSIFICATION)
# ==============================================================================
import pandas as pd

print("\n" + "=" * 65)
print("  STEP 0: DATA LOADING & RECORD INSPECTION (TASK 2)")
print("=" * 65)

# 1. Load Training and Validation Data
train_df = pd.read_json("classification_training_data.jsonl", lines=True)
val_df = pd.read_json("classification_validation_data.jsonl", lines=True)

# 2. Load Test Data
test_classification_df = pd.read_json("classification_test_data.jsonl", lines=True)

# 3. Convert DataFrames into lists of dictionaries expected by fine-tuning code
train_classification_data = train_df.to_dict("records")
val_classification_data = val_df.to_dict("records")
test_classification_data = test_classification_df.to_dict("records")

# 4. Print dataset sizes and first records for verification
print(f"Classification Training Records   : {len(train_classification_data)}")
print(f"Classification Validation Records : {len(val_classification_data)}")
print(f"Classification Test Records       : {len(test_classification_data)}\n")

print("--- Sample Train Record ---")
print(train_classification_data[0])
print("\n--- Sample Validation Record ---")
print(val_classification_data[0])
print("\n--- Sample Classification Test Record ---")
print(test_classification_data[0])
print("=" * 65 + "\n")


  STEP 0: DATA LOADING & RECORD INSPECTION (TASK 2)
Classification Training Records   : 235
Classification Validation Records : 29
Classification Test Records       : 30

--- Sample Train Record ---
{'input': "Classify the following South African medical scheme claim submission into one of the policy decision categories: 'In-Cover', 'Out-of-Cover', or 'Requires Pre-Auth'.\n\nMedical Scheme & Plan: Bestmed Pace Series (Comprehensive Plan)\nFacility: NHI Pilot Clinic (Khayelitsha, Western Cape)\nMedical Specialty: Nephrology\nClinical Description: Patient with end-stage renal disease secondary to hypertension, a reasonable candidate for a kidney transplantation.", 'output': 'In-Cover'}

--- Sample Validation Record ---
{'input': "Classify the following South African medical scheme claim submission into one of the policy decision categories: 'In-Cover', 'Out-of-Cover', or 'Requires Pre-Auth'.\n\nMedical Scheme & Plan: Bonitas BonStart (Basic Entry Network Option)\nFacility: Public Tertia

In [ ]:
# ==============================================================================
# STEP 1: DEFINE CONTROL TOKENS & TURN FORMATTING FUNCTION
# ==============================================================================
print("\n" + "=" * 65)
print("  STEP 1: GEMMA 3 CONTROL TOKENS & TURN FORMATTER SETUP")
print("=" * 65)

SOT = "<start_of_turn>"
EOT = "<end_of_turn>"

def format_io_as_turns(
    input_text: str, label_text: str, sot: str = SOT, eot: str = EOT
) -> Tuple[str, str]:
    """
    Formats a single input/output pair into user and model turn strings.

    Returns:
      formatted_q: "<start_of_turn>user\n<input_text><end_of_turn>\n"
      formatted_a: "<start_of_turn>model\n<label_text><end_of_turn>"
    """
    formatted_q = f"{sot}user\n{input_text}{eot}\n"
    formatted_a = f"{sot}model\n{label_text}{eot}"
    return formatted_q, formatted_a

# ==============================================================================
# STEP 2: LOAD JSONL PARTITIONS & PREPARE INSTRUCTION DATASETS
# ==============================================================================
print("\n" + "=" * 65)
print("  STEP 2: LOADING PREPROCESSED DATASET PARTITIONS")
print("=" * 65)

def load_partition_records(filename: str) -> Tuple[List[Dict[str, Any]], List[str], List[str]]:
    """Loads clean JSONL records and returns raw dicts, formatted prompts, and target responses."""
    if not os.path.exists(filename):
        alt_filename = filename.replace("classification_", "task2_classification_")
        if os.path.exists(alt_filename):
            filename = alt_filename
        else:
            raise FileNotFoundError(f"Missing required partition file: '{filename}'")

    raw_records = []
    inputs = []
    targets = []

    with open(filename, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                rec = json.loads(line.strip())
                q, a = format_io_as_turns(rec["input"], rec["output"])
                raw_records.append(rec)
                inputs.append(q)
                targets.append(a)

    return raw_records, inputs, targets

# Load dataset splits
train_records, train_inputs, train_targets = load_partition_records("classification_training_data.jsonl")
val_records, val_inputs, val_targets = load_partition_records("classification_validation_data.jsonl")
test_records, test_inputs, test_targets = load_partition_records("classification_test_data.jsonl")

# Format dictionary inputs for Gemma 3 Causal LM Preprocessor
train_dataset = {"prompts": train_inputs, "responses": train_targets}
val_dataset = {"prompts": val_inputs, "responses": val_targets}

print(f"✅ Data Partitions Loaded & Formatted for KerasNLP:")
print(f"  • Training   : {len(train_inputs)} instruction pairs")
print(f"  • Validation : {len(val_inputs)} instruction pairs")
print(f"  • Testing    : {len(test_inputs)} instruction pairs\n")

print("--- Sample Prompt Preview ---")
print(train_inputs[0][:200] + "...")
print("--- Sample Target Preview ---")
print(train_targets[0] + "\n")

# ==============================================================================
# STEP 3: GEMMA 3 1B MODEL LOADING & MEMORY OPTIMISATION
# ==============================================================================
print("=" * 65)
print("  STEP 3: GEMMA 3 1B MODEL LOADING & MEMORY CONFIGURATION")
print("=" * 65)

# 1. Enable mixed_bfloat16 precision policy
keras.config.set_dtype_policy("mixed_bfloat16")
keras.utils.set_random_seed(812)

# 2. Load Gemma 3 1B base model
preset_name = "gemma3_1b"
print(f"Loading preset '{preset_name}' in mixed_bfloat16 precision...")
model = keras_nlp.models.Gemma3CausalLM.from_preset(preset_name)

# 3. Set sequence length ceiling
if hasattr(model, "preprocessor") and model.preprocessor is not None:
    model.preprocessor.sequence_length = 512
    print("✅ Preprocessor sequence length set to 512 tokens.")

print("=" * 65 + "\n")


  STEP 1: GEMMA 3 CONTROL TOKENS & TURN FORMATTER SETUP

  STEP 2: LOADING PREPROCESSED DATASET PARTITIONS
✅ Data Partitions Loaded & Formatted for KerasNLP:
  • Training   : 235 instruction pairs
  • Validation : 29 instruction pairs
  • Testing    : 30 instruction pairs

--- Sample Prompt Preview ---
<start_of_turn>user
Classify the following South African medical scheme claim submission into one of the policy decision categories: 'In-Cover', 'Out-of-Cover', or 'Requires Pre-Auth'.

Medical Scheme...
--- Sample Target Preview ---
<start_of_turn>model
In-Cover<end_of_turn>

  STEP 3: GEMMA 3 1B MODEL LOADING & MEMORY CONFIGURATION
Loading preset 'gemma3_1b' in mixed_bfloat16 precision...
✅ Preprocessor sequence length set to 512 tokens.



In [ ]:
# ==============================================================================
# STEP 4: LoRA FINE-TUNING LOOP (BATCH_SIZE = 1 FOR OOM PREVENTION)
# ==============================================================================
print("=" * 65)
print("  STEP 4: LoRA FINE-TUNING SETUP & MODEL TRAINING")
print("=" * 65)

# 1. Enable LoRA rank=8 backbone adaptation
lora_rank = 8
if hasattr(model.backbone, "lora_enabled") and model.backbone.lora_enabled:
    print(f"ℹ️ LoRA adapter (rank={lora_rank}) is already enabled on Gemma 3 backbone.")
else:
    try:
        model.backbone.enable_lora(rank=lora_rank)
        print(f"✅ Enabled LoRA on Gemma 3 backbone (rank={lora_rank}).")
    except (ValueError, AttributeError) as e:
        print(f"ℹ️ LoRA adapter already active on backbone: {e}")

model.summary()

# 2. Compile model with AdamW optimizer
model.compile(
    optimizer=keras.optimizers.AdamW(learning_rate=1e-4, weight_decay=0.01),
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=[keras.metrics.SparseCategoricalAccuracy(name="accuracy")],
    jit_compile=True
)

# 3. Pass training dataset without inline validation to conserve VRAM
train_dataset = {"prompts": train_inputs, "responses": train_targets}

print("\nStarting LoRA Fine-Tuning Execution (batch_size=1)...")
epochs = 3
batch_size = 1  # Reduced from 2 to 1 to prevent VRAM allocation spikes

history = model.fit(
    x=train_dataset,
    epochs=epochs,
    batch_size=batch_size
)

print("\n✅ LoRA Fine-Tuning Complete!")
print("=" * 65 + "\n")

  STEP 4: LoRA FINE-TUNING SETUP & MODEL TRAINING
✅ Enabled LoRA on Gemma 3 backbone (rank=8).


Preprocessor: "gemma3_causal_lm_preprocessor"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                                                  ┃                                   Config ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ gemma3_tokenizer (Gemma3Tokenizer)                            │                      Vocab size: 262,144 │
└───────────────────────────────────────────────────────────────┴──────────────────────────────────────────┘

Model: "gemma3_causal_lm"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                  ┃ Output Shape              ┃         Param # ┃ Connected to               ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ padding_mask (InputLayer)     │ (None, None)              │               0 │ -                          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ token_ids (InputLayer)        │ (None, None)              │               0 │ -                          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ gemma3_backbone               │ (None, None, 1152)        │   1,001,190,528 │ padding_mask[0][0],        │
│ (Gemma3Backbone)              │                           │                 │ token_ids[0][0]            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ token_embedding               │ (None, None, 262144)      │     301,989,888 │ gemma3_backbone[0][0]      │
│ (ReversibleEmbedding)         │                           │                 │                            │
└───────────────────────────────┴───────────────────────────┴─────────────────┴────────────────────────────┘

 Total params: 1,001,190,528 (3.73 GB)

 Trainable params: 1,304,576 (4.98 MB)

 Non-trainable params: 999,885,952 (3.72 GB)


Starting LoRA Fine-Tuning Execution (batch_size=1)...
Epoch 1/3
235/235 ━━━━━━━━━━━━━━━━━━━━ 157s 538ms/step - accuracy: 0.0940 - loss: 0.0425 - sparse_categorical_accuracy: 0.7916
Epoch 2/3
235/235 ━━━━━━━━━━━━━━━━━━━━ 135s 575ms/step - accuracy: 0.0941 - loss: 0.0021 - sparse_categorical_accuracy: 0.9395
Epoch 3/3
235/235 ━━━━━━━━━━━━━━━━━━━━ 138s 587ms/step - accuracy: 0.0925 - loss: 0.0015 - sparse_categorical_accuracy: 0.9641

✅ LoRA Fine-Tuning Complete!



In [ ]:
# ==============================================================================
# STEP 5: FAST BATCHED PREDICTION GENERATION
# ==============================================================================
print("=" * 65)
print("  STEP 5: FAST BATCHED PREDICTION GENERATION ACROSS SPLITS")
print("=" * 65)

def generate_predictions_batched(
    records: list,
    model: keras_nlp.models.Gemma3CausalLM,
    output_jsonl_file: str,
    batch_size: int = 8,
    max_samples: int = None,
) -> pd.DataFrame:
    if max_samples and len(records) > max_samples:
        random.seed(812)
        records_to_process = random.sample(records, k=max_samples)
        print(f"Sampling {max_samples} records from {len(records)} total for fast evaluation.")
    else:
        records_to_process = records

    df_pred = pd.DataFrame(records_to_process)
    df_pred["model_output"] = ""

    # Build inference prompts ending with assistant turn header
    inference_prompts = [
        f"{SOT}user\n{rec['input']}{EOT}\n{SOT}model\n"
        for rec in records_to_process
    ]

    print(f"Running batched generation ({len(df_pred)} total records, batch_size={batch_size})...")

    generated_outputs = []
    for i in tqdm(range(0, len(inference_prompts), batch_size)):
        batch_prompts = inference_prompts[i : i + batch_size]
        batch_results = model.generate(batch_prompts, max_length=512)
        generated_outputs.extend(batch_results)

    df_pred["model_output"] = generated_outputs
    df_pred.to_json(output_jsonl_file, orient="records", lines=True, force_ascii=False)
    print(f"✅ Saved predictions to '{output_jsonl_file}'.")
    return df_pred

# 1. Sample 30 Training Records for Fast Evaluation
df_pred_train = generate_predictions_batched(
    train_records, model, "predictions_classification_train.jsonl", batch_size=8, max_samples=30
)

# 2. Complete Validation Set (30 records)
df_pred_val = generate_predictions_batched(
    val_records, model, "predictions_classification_val.jsonl", batch_size=8
)

# 3. Complete Test Set (30 records)
df_pred_test = generate_predictions_batched(
    test_records, model, "predictions_classification_test.jsonl", batch_size=8
)


  STEP 5: FAST BATCHED PREDICTION GENERATION ACROSS SPLITS
Sampling 30 records from 235 total for fast evaluation.
Running batched generation (30 total records, batch_size=8)...


100%|██████████| 4/4 [01:54<00:00, 28.54s/it]


✅ Saved predictions to 'predictions_classification_train.jsonl'.
Running batched generation (29 total records, batch_size=8)...


100%|██████████| 4/4 [00:50<00:00, 12.65s/it]


✅ Saved predictions to 'predictions_classification_val.jsonl'.
Running batched generation (30 total records, batch_size=8)...


100%|██████████| 4/4 [00:16<00:00,  4.09s/it]


✅ Saved predictions to 'predictions_classification_test.jsonl'.


In [ ]:
# ==============================================================================
# STEP 6: ACCURATE SUBSTRING LABEL EXTRACTION & EVALUATION METRICS
# ==============================================================================
print("\n" + "=" * 65)
print("  STEP 6: METRICS EVALUATION & COMPARATIVE PERFORMANCE")
print("=" * 65)

def extract_prediction(model_output: str) -> str:
    """Extracts target category from model response turn."""
    if not isinstance(model_output, str):
        return "Unknown"

    # Isolate text after assistant turn start token
    if f"{SOT}model" in model_output:
        gen_text = model_output.rsplit(f"{SOT}model", 1)[-1]
    else:
        gen_text = model_output

    gen_text = gen_text.replace(EOT, "").strip()
    gen_lower = gen_text.lower()

    # NOTE: Check Out-of-Cover BEFORE In-Cover to avoid substring collision
    if "requires pre-auth" in gen_lower or "pre-auth" in gen_lower:
        return "Requires Pre-Auth"
    elif "out-of-cover" in gen_lower or "out of cover" in gen_lower:
        return "Out-of-Cover"
    elif "in-cover" in gen_lower or "in cover" in gen_lower:
        return "In-Cover"

    return "Unknown"

def evaluate_predictions_df(df_input: pd.DataFrame, title: str) -> dict:
    df_eval = df_input.copy()
    df_eval["predicted"] = df_eval["model_output"].apply(extract_prediction)

    print(f"\n--- {title} ---")
    y_true = df_eval["output"].astype(str)
    y_pred = df_eval["predicted"].astype(str)

    accuracy = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred, average="weighted", zero_division=0)
    recall = recall_score(y_true, y_pred, average="weighted", zero_division=0)

    print(f"Accuracy            : {accuracy:.2%}")
    print(f"Precision (Weighted): {precision:.2%}")
    print(f"Recall (Weighted)   : {recall:.2%}\n")

    print("Classification Report:")
    print(classification_report(y_true, y_pred, zero_division=0))

    return {
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "df": df_eval,
    }

results_train = evaluate_predictions_df(df_pred_train, "Training Partition Performance")
results_val = evaluate_predictions_df(df_pred_val, "Validation Partition Performance")
results_test = evaluate_predictions_df(df_pred_test, "Test Partition Performance")

# Summary Comparison Table
print("\n" + "=" * 65)
print("  TASK 2 SUMMARY: SEEN VS UNSEEN PERFORMANCE COMPARISON")
print("=" * 65)
print(f"{'Metric':<20} {'Train':<15} {'Validation':<15} {'Test':<15}")
print("-" * 65)
print(f"{'Accuracy':<20} {results_train['accuracy']:.2%}{'':<7} {results_val['accuracy']:.2%}{'':<7} {results_test['accuracy']:.2%}")
print(f"{'Precision':<20} {results_train['precision']:.2%}{'':<7} {results_val['precision']:.2%}{'':<7} {results_test['precision']:.2%}")
print(f"{'Recall':<20} {results_train['recall']:.2%}{'':<7} {results_val['recall']:.2%}{'':<7} {results_test['recall']:.2%}")
print("=" * 65 + "\n")


  STEP 6: METRICS EVALUATION & COMPARATIVE PERFORMANCE

--- Training Partition Performance ---
Accuracy            : 86.67%
Precision (Weighted): 90.48%
Recall (Weighted)   : 86.67%

Classification Report:
                   precision    recall  f1-score   support

         In-Cover       0.71      1.00      0.83        10
     Out-of-Cover       1.00      1.00      1.00        10
Requires Pre-Auth       1.00      0.60      0.75        10

         accuracy                           0.87        30
        macro avg       0.90      0.87      0.86        30
     weighted avg       0.90      0.87      0.86        30


--- Validation Partition Performance ---
Accuracy            : 65.52%
Precision (Weighted): 82.76%
Recall (Weighted)   : 65.52%

Classification Report:
                   precision    recall  f1-score   support

         In-Cover       0.50      1.00      0.67        10
     Out-of-Cover       1.00      0.78      0.88         9
Requires Pre-Auth       1.00      0.20      0.

## 7. End-to-end  conversation-classification pipeline:  Care2ClaimZA-LLM

Run the following code block for a demonstration of potential use of the LLM by a patient



In [ ]:
# ==============================================================================
# CARE2CLAIMLLM-ZA: END-TO-END HYBRID INFERENCE PIPELINE
# ==============================================================================
#import re

def parse_scheme_and_claim_info(dialogue_history: str, triage_response: str) -> dict:
    """
    Extracts clinical scheme metadata, plan, and procedure details
    from dialogue history and triage guidance.
    """
    # Standardize text for extraction
    full_text = f"{dialogue_history}\n{triage_response}"

    # 1. Scheme & Plan Extraction (with sensible defaults for SA schemes)
    scheme_match = re.search(r"(Discovery|Fedhealth|Bonitas|Momentum|Medihelp)", full_text, re.IGNORECASE)
    scheme = scheme_match.group(1) if scheme_match else "Discovery Health"

    plan_match = re.search(r"(KeyCare|flexiFED|Classic|Saver|Access|Essential)", full_text, re.IGNORECASE)
    plan = plan_match.group(1) if plan_match else "Standard Option"

    # 2. Extract clinical scenario / symptoms
    symptom_match = re.search(r"(chest pain|cough|fever|headache|fracture|appendicitis|consultation|scans|mri)", full_text, re.IGNORECASE)
    clinical_scenario = symptom_match.group(0) if symptom_match else "General medical consultation and assessment"

    return {
        "scheme": scheme,
        "plan": plan,
        "clinical_scenario": clinical_scenario
    }

def run_hybrid_care2claim_pipeline(
    patient_dialogue: str,
    model_task1: keras_nlp.models.Gemma3CausalLM,
    model_task2: keras_nlp.models.Gemma3CausalLM
) -> dict:
    """
    Runs patient intake through Task 1 (Triage) and auto-routes
    the clinical payload into Task 2 (Claim Policy Classification).
    """
    print("=" * 65)
    print("  CARE2CLAIMLLM-ZA: HYBRID CONVERSATIONAL ADJUDICATION ENGINE")
    print("=" * 65)

    # --------------------------------------------------------------------------
    # STAGE 1: TASK 1 CONVERSATIONAL TRIAGE & INTAKE GUIDANCE
    # --------------------------------------------------------------------------
    prompt_task1 = f"<start_of_turn>user\n{patient_dialogue}<end_of_turn>\n<start_of_turn>model\n"
    raw_triage_output = model_task1.generate(prompt_task1, max_length=512)

    # Clean output
    triage_guidance = raw_triage_output.rsplit("<start_of_turn>model\n", 1)[-1].replace("<end_of_turn>", "").strip()

    print("\n--- STAGE 1: CONVERSATIONAL TRIAGE GUIDANCE (TASK 1) ---")
    print(triage_guidance)

    # --------------------------------------------------------------------------
    # STAGE 2: PAYLOAD EXTRACTION & TASK 2 INSTRUCTION SYNTHESIS
    # --------------------------------------------------------------------------
    extracted_meta = parse_scheme_and_claim_info(patient_dialogue, triage_guidance)

    formatted_claim_input = (
        f"Medical Scheme: {extracted_meta['scheme']} ({extracted_meta['plan']}). "
        f"Patient presenting with: {extracted_meta['clinical_scenario']}."
    )

    print("\n--- STAGE 2: AUTO-FORMATTED TASK 2 CLAIM PAYLOAD ---")
    print(formatted_claim_input)

    # --------------------------------------------------------------------------
    # STAGE 3: TASK 2 POLICY CLASSIFICATION ADJUDICATION
    # --------------------------------------------------------------------------
    prompt_task2 = f"<start_of_turn>user\n{formatted_claim_input}<end_of_turn>\n<start_of_turn>model\n"
    raw_class_output = model_task2.generate(prompt_task2, max_length=512)

    raw_category = raw_class_output.rsplit("<start_of_turn>model\n", 1)[-1].replace("<end_of_turn>", "").strip()

    # Map to standard policy category
    if "requires pre-auth" in raw_category.lower():
        policy_decision = "Requires Pre-Auth"
    elif "out-of-cover" in raw_category.lower():
        policy_decision = "Out-of-Cover"
    else:
        policy_decision = "In-Cover"

    print("\n--- STAGE 3: POLICY ADJUDICATION DECISION (TASK 2) ---")
    print(f"Final Policy Status: [{policy_decision}]")
    print("=" * 65 + "\n")

    return {
        "triage_guidance": triage_guidance,
        "claim_payload": formatted_claim_input,
        "policy_decision": policy_decision
    }

In [ ]:
# ==============================================================================
# TEST INVOCATION: RUN A SAMPLE PATIENT DIALOGUE THROUGH HYBRID PIPELINE
# ==============================================================================

# 1. Define sample patient intake dialogue
sample_patient_dialogue = """
Sawubona doc! I have a severe burning pain in my chest when I cough.
I am a 29-year-old female on Discovery KeyCare Access, currently at Netcare Park Lane.
"""

# 2. Execute end-to-end hybrid pipeline
pipeline_output = run_hybrid_care2claim_pipeline(
    patient_dialogue=sample_patient_dialogue,
    model_task1=model,  # Passes Task 1 model instance
    model_task2=model   # Passes Task 2 model instance
)

  CARE2CLAIMLLM-ZA: HYBRID CONVERSATIONAL ADJUDICATION ENGINE

--- STAGE 1: CONVERSATIONAL TRIAGE GUIDANCE (TASK 1) ---
FhirExtension

--- STAGE 2: AUTO-FORMATTED TASK 2 CLAIM PAYLOAD ---
Medical Scheme: Discovery (KeyCare). Patient presenting with: cough.

--- STAGE 3: POLICY ADJUDICATION DECISION (TASK 2) ---
Final Policy Status: [In-Cover]

